# NICE on PROMISE — conference-compute edition

This is a **separate, compute-conscious copy** of the supplied v3 notebook. The original file is not modified. It preserves the project-wise holdout protocol, validation-only model selection, teacher/student knowledge distillation, a no-KD ablation, INT8-safe quantization, leakage study, per-class/error analysis, and export/reporting.

## External services
The **only external API** used by this notebook is **Groq** (LLM-written explanation targets, and the optional LLM judge). The **only file you upload is your dataset CSV**; no trained weights or backups are uploaded, and no dataset is downloaded for you. (Pretrained base checkpoints such as `google/flan-t5-*` and `bert-base-uncased` are still fetched from the Hugging Face Hub as ordinary model downloads; they are not LLM API calls.)

## Important scope statement
- The study uses **one single project-wise train/validation/test split** (about 20% of projects test, about 20% of the rest validation, remainder train; `SPLIT` in the config cell). There are no folds and no cross-validation for the neural models. The only 5-fold step is the cheap TF-IDF leakage study (`LEAK_FOLDS`).
- Only training seed 42 is used (enforced by an assert) in this economical first run. This does **not** estimate seed variability.
- This notebook is for a first conference-result run. Before making strong claims, repeat the primary comparison with at least 3 seeds and/or run all folds if compute permits, and use an independent dataset for external generalization.
- No test-set result is used for model selection. Do not change the pre-specified selection margin after looking at test results.
- Explanation targets remain LLM-generated targets; their presence does not prove that generated explanations are faithful or correct. Human evaluation is needed for explanation-quality claims.

## Recommended run order
1. Use a GPU runtime with Internet enabled (Kaggle: Settings -> Accelerator + Internet); run cells top-to-bottom. Add your CSV as a Kaggle input first (it is required; there is no download fallback).
2. Keep your Groq key in Kaggle Secrets (Add-ons -> Secrets) or Colab Secrets under the name `GROQ_API_KEY` (never hard-code it, and never paste the key itself where the name goes). Groq is the only LLM API used.
3. Models are always trained in the current session; there is no option to upload earlier weights.
4. If the runtime disconnects, reconnect and rerun from the top. Within a still-running session, completed explanation batches and fold artifacts are cached in `nice_conference_efficient/`.
5. After the first run, inspect the generated `export/` tables and protocol report. Do not describe this single split as cross-validation.

## Compute trade-off
Compared with the original standard configuration, this edition reduces the number of candidate models, LR trials, epochs, quantizers, bootstrap replicates, optional Groq-based LLM judging, explanation-free ablation, and final refit. It keeps the central compression comparison (T5-base vs T5-small KD/no-KD, plus INT8-safe). These settings reduce compute but are not a substitute for multi-seed/multi-fold validation.


> **Revision notes:** API key removed (rotate it); `DATA_MD5_OK` fixed; 5 folds x 3 seeds enabled; see ANALYSIS.md.


## Estimated run time (first run from scratch, Colab T4 GPU, ~970 requirements)

These are estimates, not measurements; actual time depends on GPU, dataset size and Groq rate limits.

| Stage | Approx. time |
|---|---|
| Install + model downloads | 5-10 min |
| Groq explanation targets (rate-limited) | 30-40 min |
| Baseline flan-t5-base (6 epochs) | 8-12 min |
| Teacher flan-t5-large (5 epochs) | 25-40 min |
| Student T5-small with KD (20 epochs) | 15-25 min |
| Student T5-small without KD | 8-12 min |
| BERT + TF-IDF baselines | 3-5 min |
| INT8 quantization + val/test predictions | 10-20 min |
| Leakage study, bootstrap CIs, ablations | 5-8 min |
| BERTScore, latency, export | 8-12 min |
| **Total** | **~2.5-3.5 h (A100/L4: ~1-1.5 h; CPU only: impractical)** |

**Ways to save time:** reuse cached `explanations_*.json` when you re-run in the same session (saves 30-40 min).

**Risks:**
- Groq free tier allows ~200k tokens/day; if the explanation cell stops with a quota error, rerun it after the UTC midnight reset (finished batches are cached).
- Free Colab can disconnect after a few hours and the local disk is then lost, so a new session retrains from scratch (only your CSV has to be uploaded again).
- If training shows `loss: nan` (T5 in fp16, mainly flan-t5-large), set `fp16=False` in the three training-argument blocks.
- Human evaluation (two annotators) is not included in these times.

In [1]:
!pip -q install "transformers>=4.44" datasets "accelerate>=0.33" bitsandbytes sentencepiece scikit-learn rouge_score groq bert_score pandas matplotlib tqdm scipy

In [2]:
import os, re, json, random, hashlib, time, math, shutil, gc, pickle, platform, sys, copy, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ====================================================================== CONFIG
PRESET = "conference_efficient"
SEED = 42
WORK_DIR = "nice_conference_efficient"; os.makedirs(WORK_DIR, exist_ok=True)
# Reuse prior explanation cache if present; cache files are prompt-hash checked by the original code.
for _old_dir in ("nice_work_v2", "nice_work_v3"):
    if os.path.isdir(_old_dir):
        for _f in os.listdir(_old_dir):
            if _f.startswith("explanations_") and not os.path.exists(os.path.join(WORK_DIR, _f)):
                shutil.copy(os.path.join(_old_dir, _f), WORK_DIR)

CFG = dict(
    RUN_FOLDS=[0, 1, 2, 3, 4],                  # one single project-wise train/validation/test split (id 0); no cross-validation
    BASE_EPOCHS=6, TEACHER_EPOCHS=5, STUDENT_EPOCH_SCALE=1.0,
    QUANT_KINDS=("int8-safe",), EVAL_BEAMS=1, BOOT_N=1000, ENC_EPOCHS=15,
    TRAIN_SEEDS=(42, 43, 44), BASE_LR_GRID=(3e-4,), STUDENT_LR_MULTS=(1.0,)
)
QUANT_KINDS, EVAL_BEAMS, BOOT_N, ENC_EPOCHS = (CFG[k] for k in
    ("QUANT_KINDS", "EVAL_BEAMS", "BOOT_N", "ENC_EPOCHS"))
# single split by whole projects: ~20% of projects = TEST; ~20% of the remaining projects = VALIDATION (carved out inside run_fold); rest = TRAIN
SPLIT = dict(TEST_FRAC=0.2, VAL_FRAC=0.2)
LEAK_FOLDS = 5   # used ONLY by the cheap TF-IDF leakage study (section 6); not used for training/evaluating the neural models

# ---- data
DATA_MD5 = "977399184e55483b3c971b7094a53279"
DATA_PATH = os.path.join(WORK_DIR, "PROMISE-relabeled-NICE.csv")
DROP_UNLABELED = False

# ---- validation protocol (preserved)
INNER_VAL_GROUP_FRAC = SPLIT["VAL_FRAC"]
NEAR_DUP_THRESH = 0.80
OVERSAMPLE_RARE, MIN_CLASS_COUNT = True, 30

# ---- models: one compact student instead of three
BASELINE_NAME = "google/flan-t5-base"
TEACHER_NAME  = "google/flan-t5-large"
STUDENTS = [("google/flan-t5-small", 5e-4, 20)]
BASE_EPOCHS, BASE_LR = CFG["BASE_EPOCHS"], 3e-4
TEACHER_EPOCHS, TEACHER_LR = CFG["TEACHER_EPOCHS"], 3e-4
KD_ALPHA, KD_T = 0.5, 2.0
TRY_NOKD_ABLATION = True
MAX_IN_LEN, MAX_OUT_LEN = 192, 192
RUN_ENCODER_BASELINE, ENCODER_NAME, ENC_LR, ENC_BS = True, "bert-base-uncased", 5e-5, 16

# ---- statistics / selection: specify before viewing test results
NONINF_DELTA = 0.03
BOOT_UNIT = "project"
KEEP_MODELS = True

# ---- explanations: Groq is the ONLY external API (explanation targets + optional judge)
USE_GROQ = True
ALLOW_TEMPLATE_EXPLANATIONS = False
RUN_LLM_JUDGE, JUDGE_N = False, 0           # optional LLM judge (also runs on Groq)
JUDGE_MODEL = None                          # None = auto-pick a Groq model that differs from the explanation writer
LAT_N = 20

# ---- optional/high-cost stages
TRAIN_SEEDS = list(CFG["TRAIN_SEEDS"])
assert len(TRAIN_SEEDS) >= 1 and all(isinstance(x, int) for x in TRAIN_SEEDS)  # use (42, 43, 44) for the paper run
BASE_LR_GRID = tuple(CFG["BASE_LR_GRID"])
STUDENT_LR_MULTS = tuple(CFG["STUDENT_LR_MULTS"])
TUNE_STUDENTS = False
RUN_NOEXPL_ABLATION = False
RUN_LEAKAGE_STUDY = True
REFIT_FINAL = False
HUMAN_EVAL_DIR = os.path.join(WORK_DIR, "human_eval")
EXTERNAL_DATASET = dict(path=None, name="second-dataset", text_col="text", group_col=None, fr_col="is_functional", q_col="is_quality",
                        class_cols={}, class_col=None, class_sep=";", class_value_map={})

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()
# Safe speed/memory settings for supported NVIDIA GPUs; no change to the model/data protocol.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

NFR_CLASSES = [("Availability","A"),("Fault Tolerance","FT"),("Legal","L"),("Look & Feel","LF"),("Maintainability","MN"),
               ("Operability","O"),("Performance","PE"),("Portability","PO"),("Scalability","SC"),("Security","SE"),
               ("Usability","US"),("Other","OT")]
def norm(s): return re.sub(r"[^a-z0-9]", "", str(s).lower())

ENV = dict(python=sys.version.split()[0], torch=torch.__version__, platform=platform.platform(),
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
for pkg in ("transformers", "datasets", "accelerate", "bitsandbytes", "sklearn", "pandas", "numpy", "rouge_score"):
    try: ENV[pkg] = __import__(pkg).__version__
    except Exception: ENV[pkg] = "n/a"
print(PRESET, "| device:", ENV["gpu"]); print(ENV)
if DEVICE == "cpu": print("WARNING: no GPU - training will be extremely slow; use a Colab GPU runtime.")
print("Run plan:", CFG, "| students:", STUDENTS, "| optional stages: judge=False, no-explanation=False, refit=False")


conference_efficient | device: Tesla T4
{'python': '3.13.16', 'torch': '2.11.0+cu130', 'platform': 'Linux-6.18.55+-x86_64-with-glibc2.39', 'gpu': 'Tesla T4', 'transformers': '5.20.2', 'datasets': '4.8.5', 'accelerate': '1.15.0', 'bitsandbytes': '0.50.2', 'sklearn': '1.6.1', 'pandas': '2.3.3', 'numpy': '2.1.3', 'rouge_score': 'n/a'}
Run plan: {'RUN_FOLDS': [0, 1, 2, 3, 4], 'BASE_EPOCHS': 6, 'TEACHER_EPOCHS': 5, 'STUDENT_EPOCH_SCALE': 1.0, 'QUANT_KINDS': ('int8-safe',), 'EVAL_BEAMS': 1, 'BOOT_N': 1000, 'ENC_EPOCHS': 15, 'TRAIN_SEEDS': (42, 43, 44), 'BASE_LR_GRID': (0.0003,), 'STUDENT_LR_MULTS': (1.0,)} | students: [('google/flan-t5-small', 0.0005, 20)] | optional stages: judge=False, no-explanation=False, refit=False


## 2. Data upload (CSV) + audit + project-wise folds

**On Kaggle: add your PROMISE/NICE CSV through the right sidebar (Add Input -> Upload -> Create).** Cell A finds it under `/kaggle/input`, copies it to `DATA_PATH`, and Cell B loads it, checks the required columns and defines `df` and `LABELS`. There is no download fallback: if no `.csv` is found, Cell A stops with an error.

Expected schema: a requirement text column (`RequirementText`, `Requirement`, or `Text`), `IsFunctional`, `IsQuality`, `ProjectID`, and the 12 NICE NFR category columns (full names or abbreviations). If your CSV uses different column names or a different label scheme, update the mapping in the data-loading cell before training. ProjectID is required for the project-wise split used in this study.

In [3]:
# ===== Cell A: locate the uploaded CSV in /kaggle/input and copy it to DATA_PATH
import os, glob, shutil

os.makedirs(WORK_DIR, exist_ok=True)
DATA_PATH = os.path.join(WORK_DIR, "PROMISE-relabeled-NICE.csv")
UPLOAD_DONE = False

csv_files = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
if not csv_files:
    raise FileNotFoundError(
        "No CSV found in /kaggle/input.\n"
        "Right sidebar -> 'Add Input' -> 'Upload' -> add your CSV -> Create, then re-run this cell."
    )

PICK_NAME = None   # e.g. "promise" -> pick the CSV whose filename contains this text

if PICK_NAME:
    matches = [p for p in csv_files if PICK_NAME.lower() in os.path.basename(p).lower()]
else:
    matches = csv_files

if not matches:
    raise FileNotFoundError(f"No CSV matching '{PICK_NAME}'. Found: {csv_files}")
if len(matches) > 1:
    print("Multiple CSVs found; using the first. Set PICK_NAME to choose another:")
    for p in matches:
        print("  ", p)

shutil.copy(matches[0], DATA_PATH)
UPLOAD_DONE = True
print(f"Copied {matches[0]} -> {DATA_PATH} ({os.path.getsize(DATA_PATH):,} bytes)")

Copied /kaggle/input/datasets/sadiaazam9999999o/sadia-dataset/PROMISE-relabeled-NICE.csv -> nice_conference_efficient/PROMISE-relabeled-NICE.csv (95,725 bytes)


In [4]:
# ===== Cell B: load CSV, validate schema, build canonical df + LABELS
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

_need = [n for n in ("WORK_DIR", "DATA_PATH", "NFR_CLASSES", "norm", "DROP_UNLABELED") if n not in globals()]
if _need:
    raise NameError(f"Run the config cell and Cell A first. Undefined: {_need}")
if not globals().get("UPLOAD_DONE") or not os.path.exists(DATA_PATH):
    raise RuntimeError("No CSV at DATA_PATH. Run Cell A first.")

# Optional integrity check (warning only)
DATA_MD5_OK = None
if "DATA_MD5" in globals():
    _md5 = hashlib.md5(open(DATA_PATH, "rb").read()).hexdigest()
    DATA_MD5_OK = (_md5 == DATA_MD5)
    if _md5 != DATA_MD5:
        print(f"NOTE: CSV md5 {_md5} differs from the expected DATA_MD5 {DATA_MD5}; this is not the reference file.")

# ---- Read CSV with common encodings
raw, last_read_error = None, None
for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
    try:
        raw = pd.read_csv(DATA_PATH, encoding=enc)
        print("CSV encoding:", enc, "| shape:", raw.shape)
        break
    except UnicodeDecodeError as e:
        last_read_error = e
if raw is None:
    raise RuntimeError(f"Could not decode the CSV. Last error: {last_read_error}")
if raw.empty:
    raise ValueError("The uploaded CSV contains no data rows.")

# ---- Normalise typographic characters in text-like columns
_REPL = {"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u00a0": " "}
for c in raw.select_dtypes(include=["object", "string"]).columns:
    s = raw[c].astype("string")
    for a, b in _REPL.items():
        s = s.str.replace(a, b, regex=False)
    raw[c] = s

# ---- Column resolution
def find_col(d, full, abbr):
    cols = {norm(c): c for c in d.columns}
    for key in (norm(f"{full} ({abbr})"), norm(full), norm(abbr)):
        if key in cols:
            return cols[key]
    for k, c in cols.items():
        if k.startswith(norm(full)):
            return c
    raise KeyError(f"Missing NICE category column '{full}' (or '{abbr}'). Available columns: {list(d.columns)}")

_norm_cols = {norm(c): c for c in raw.columns}
text_col = next((_norm_cols[k] for k in ("requirementtext", "requirement", "text") if k in _norm_cols), None)
fr_col   = _norm_cols.get("isfunctional")
q_col    = _norm_cols.get("isquality")
proj_col = _norm_cols.get("projectid")
missing_core = [name for name, value in (("RequirementText/Requirement/Text", text_col),
                                         ("IsFunctional", fr_col), ("IsQuality", q_col),
                                         ("ProjectID", proj_col)) if value is None]
if missing_core:
    raise ValueError(
        "Your CSV does not match the expected NICE/PROMISE schema. Missing: " + ", ".join(missing_core) +
        "\nAvailable columns: " + ", ".join(map(str, raw.columns)) +
        "\nPlease rename/map these columns in this cell before continuing."
    )

category_col_map = {full: find_col(raw, full, abbr) for full, abbr in NFR_CLASSES}
_dups = [c for c in set(category_col_map.values()) if list(category_col_map.values()).count(c) > 1]
if _dups:
    raise ValueError(f"Several NFR classes resolved to the same CSV column: {_dups}. Check the column names.")

# ---- Binary conversion
_BIN_MAP = {"1": 1, "1.0": 1, "true": 1, "yes": 1, "y": 1,
            "0": 0, "0.0": 0, "false": 0, "no": 0, "n": 0}

def to_binary(series, column_name, na_as_zero=False):
    """Convert common binary encodings to 0/1; reject ambiguous values. Truly empty cells -> 0 only if na_as_zero."""
    orig_na = series.isna()
    if pd.api.types.is_numeric_dtype(series) or pd.api.types.is_bool_dtype(series):
        vals = pd.to_numeric(series, errors="coerce")
    else:
        s = series.astype("string").str.strip().str.lower()
        vals = pd.to_numeric(s.map(_BIN_MAP), errors="coerce")
    if na_as_zero:
        vals = vals.mask(orig_na, 0)
    bad = int(vals.isna().sum())
    if bad:
        raise ValueError(f"Column '{column_name}' has {bad} missing/non-binary values. Clean or map them before training.")
    unique = set(vals.unique().tolist())
    if not unique.issubset({0, 1}):
        raise ValueError(f"Column '{column_name}' must contain only 0/1 (or yes/no, true/false). Found: {sorted(unique)[:10]}")
    return vals.astype(int)

# ---- Validate project IDs before building the frame
_proj = raw[proj_col].astype("string").str.strip()
_bad_proj = _proj.isna() | _proj.str.lower().isin(["", "nan", "<na>", "none"])
if _bad_proj.any():
    raise ValueError(f"ProjectID has {int(_bad_proj.sum())} missing/blank values. Project-wise evaluation requires valid project IDs.")

# ---- Canonical dataframe expected by the rest of the notebook
_text = raw[text_col].fillna("").astype(str).str.strip().str.strip("'\"").str.strip()
df = pd.DataFrame({
    "project": _proj.astype(str).values,
    "text": _text.values,
    "is_functional": to_binary(raw[fr_col], fr_col).values,
    "is_quality": to_binary(raw[q_col], q_col).values,
})
for full, _abbr in NFR_CLASSES:
    df[full] = to_binary(raw[category_col_map[full]], category_col_map[full], na_as_zero=True).values

# ---- Remove empty / duplicate requirements
_blank = int((df.text.str.strip() == "").sum())
if _blank:
    print("Warning:", _blank, "blank requirement texts will be removed.")

df["_key"] = (df.text.str.lower()
                .str.replace(r"[^a-z0-9 ]", " ", regex=True)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip())
n0 = len(df)
df = df[df._key.str.len() > 0].drop_duplicates(subset="_key").reset_index(drop=True)
print(f"Removed {n0 - len(df)} duplicate/empty requirements.")

# ---- LABELS (drop classes with no examples)
LABELS = [full for full, _ in NFR_CLASSES]
empty = [l for l in LABELS if df[l].sum() == 0]
if empty:
    print("Dropping classes with 0 examples:", empty)
LABELS = [l for l in LABELS if l not in empty]
if not LABELS:
    raise ValueError("No NFR class has any positive example; check the category columns.")

# ---- Data audit
neither = (df.is_functional == 0) & (df.is_quality == 0)
q_no_cls = (df.is_quality == 1) & (df[LABELS].sum(axis=1) == 0)
cls_no_q = (df.is_quality == 0) & (df[LABELS].sum(axis=1) > 0)
print(f"AUDIT: neither FR nor NFR flag: {int(neither.sum())} | quality=1 but no NFR class: {int(q_no_cls.sum())} | "
      f"NFR class but quality=0: {int(cls_no_q.sum())}")
if neither.any():
    display(df.loc[neither, ["project", "text"]].head(10))
if DROP_UNLABELED:
    df = df[~neither].reset_index(drop=True)
    print("Dropped unlabeled rows ->", len(df))

print("Requirements:", len(df), "| projects:", df.project.nunique(), "| classes:", len(LABELS))
print("Functional:", int(df.is_functional.sum()), "| Quality:", int(df.is_quality.sum()),
      "| both:", int(((df.is_functional == 1) & (df.is_quality == 1)).sum()))
print("Avg #NFR labels per quality requirement: %.2f" % df.loc[df.is_quality == 1, LABELS].sum(axis=1).mean())
print("LABELS:", LABELS)

CSV encoding: utf-8 | shape: (622, 16)
Removed 0 duplicate/empty requirements.
Dropping classes with 0 examples: ['Other']
AUDIT: neither FR nor NFR flag: 13 | quality=1 but no NFR class: 0 | NFR class but quality=0: 0


,project,text
13,1,The data displayed in both the nodes within th...
111,3,A class shall be either a non-clinical class o...
112,3,A non-clinical class shall specify the course ...
113,3,A clinical class shall specify the course name...
115,3,A Program of Study shall consist of a program ...
121,3,Clinical site information shall include (but n...
123,3,A clinical lab section shall include the clini...
170,4,All updates to data files or database must be ...
240,5,The vehicle data shall include vehicle year ma...
241,5,The vehicle location shall include street addr...


Requirements: 622 | projects: 15 | classes: 11
Functional: 308 | Quality: 381 | both: 80
Avg #NFR labels per quality requirement: 1.36
LABELS: ['Availability', 'Fault Tolerance', 'Legal', 'Look & Feel', 'Maintainability', 'Operability', 'Performance', 'Portability', 'Scalability', 'Security', 'Usability']


In [5]:
# ===================================================== pure helpers: format, parsing, metrics, bootstrap, folds
LABEL_LOOKUP = {norm(l): l for l in LABELS}
INSTR = "Identify whether the software requirement is functional and/or quality (non-functional), classify its NFR categories, and explain."
def make_input(text): return f"{INSTR}\nRequirement: {text}"

def pred_string(fr, q, cats, expl=""):
    c = "; ".join(cats) if len(cats) else "none"
    return f"functional: {'yes' if fr else 'no'} | quality: {'yes' if q else 'no'} | categories: {c} | explanation: {expl}"

def make_target(row):
    return pred_string(row["is_functional"], row["is_quality"], [l for l in LABELS if row[l] == 1], row["explanation"])

_RE_FR = re.compile(r"functional:\s*(yes|no)\b", re.I)
_RE_Q  = re.compile(r"\bquality:\s*(yes|no)\b", re.I)
_RE_C  = re.compile(r"categories:\s*([^|]*)", re.I)
_RE_E  = re.compile(r"explanation:\s*(.*)", re.I | re.S)
def parse_output(s):
    """dict(fr, q, cats, expl, ok). Malformed output -> all-zero prediction with ok=False (counted as errors)."""
    s = (s or "").strip()
    mf, mq, mc, me = _RE_FR.search(s), _RE_Q.search(s), _RE_C.search(s), _RE_E.search(s)
    expl = me.group(1).strip() if me else ""
    if not (mf and mq and mc): return dict(fr=0, q=0, cats=set(), expl=expl, ok=False)
    cats = set()
    for c in re.split(r"[;,]", mc.group(1)):
        k = norm(c)
        if k in LABEL_LOOKUP: cats.add(LABEL_LOOKUP[k])
    return dict(fr=int(mf.group(1).lower() == "yes"), q=int(mq.group(1).lower() == "yes"), cats=cats, expl=expl, ok=True)

def parsed_to_arrays(parsed):
    fr = np.array([p["fr"] for p in parsed], dtype=int); q = np.array([p["q"] for p in parsed], dtype=int)
    P = np.array([[int(l in p["cats"]) for l in LABELS] for p in parsed], dtype=int).reshape(len(parsed), len(LABELS))
    return fr, q, P
def gold_arrays(d): return d.is_functional.values.astype(int), d.is_quality.values.astype(int), d[LABELS].values.astype(int)
def arrays_to_raw(fr, q, P): return [pred_string(a, b, [l for l, v in zip(LABELS, p) if v]) for a, b, p in zip(fr, q, P)]

# ---- explanation label-consistency: does the explanation name exactly the predicted categories?
_STEMS = {"Availability": r"availab", "Fault Tolerance": r"fault[\s-]*toleran", "Legal": r"\blegal", "Look & Feel": r"look\s*(?:&|and)\s*feel",
          "Maintainability": r"maintainab", "Operability": r"operab", "Performance": r"performance", "Portability": r"portab",
          "Scalability": r"scalab", "Security": r"secur", "Usability": r"usab", "Other": r"\bother\b"}
CAT_RE = {l: re.compile(_STEMS[l], re.I) for l in LABELS}
def mentioned_cats(expl): return {l for l, r in CAT_RE.items() if r.search(expl or "")}
def label_consistency(expl, cats): return mentioned_cats(expl) == set(cats)

# ---- metrics (numpy, fast enough for bootstrap). gold/pred = (fr, q, Y) arrays
def metrics_np(gold, pred, idx=None):
    gfr, gq, gY = gold; pfr, pq, pY = pred
    if idx is not None: gfr, gq, gY, pfr, pq, pY = gfr[idx], gq[idx], gY[idx], pfr[idx], pq[idx], pY[idx]
    out = {}
    for tag, y, p in (("FR", gfr, pfr), ("NFR", gq, pq)):
        tp = float((y * p).sum()); fp = float(((1 - y) * p).sum()); fn = float((y * (1 - p)).sum())
        out[f"ident_{tag}_F1"] = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
    tp = (gY * pY).sum(0).astype(float); fp = ((1 - gY) * pY).sum(0).astype(float); fn = (gY * (1 - pY)).sum(0).astype(float)
    d = 2 * tp + fp + fn; per = np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0); sup = gY.sum(0); m = sup > 0
    D = 2 * tp.sum() + fp.sum() + fn.sum()
    out["cls_micro_F1"] = float(2 * tp.sum() / D) if D > 0 else 0.0
    out["cls_macro_F1"] = float(per[m].mean()) if m.any() else 0.0           # classes with support only
    out["cls_weighted_F1"] = float((per * sup).sum() / sup.sum()) if sup.sum() > 0 else 0.0
    rows = gY.sum(1) > 0
    out["cls_exact_match_nfr"] = float((gY[rows] == pY[rows]).all(1).mean()) if rows.any() else 0.0   # NFR rows only
    out["cls_hamming_loss"] = float((gY != pY).mean())
    out["score"] = float(np.mean([out["ident_FR_F1"], out["ident_NFR_F1"], out["cls_micro_F1"], out["cls_macro_F1"]]))
    return out

def score_raw(raw, d):
    parsed = [parse_output(r) for r in raw]
    m = metrics_np(gold_arrays(d), parsed_to_arrays(parsed)); m["format_ok"] = float(np.mean([p["ok"] for p in parsed])); return m

def per_class_f1(gold, pred, idx=None):
    _, _, gY = gold; _, _, pY = pred
    if idx is not None: gY, pY = gY[idx], pY[idx]
    tp = (gY * pY).sum(0).astype(float); fp = ((1 - gY) * pY).sum(0); fn = (gY * (1 - pY)).sum(0); d = 2 * tp + fp + fn
    f1 = np.where(d > 0, 2 * tp / np.maximum(d, 1), 0.0); sup = gY.sum(0)
    return np.where(sup > 0, f1, np.nan), sup

def per_class_table(gold, pred):
    _, _, gY = gold; _, _, pY = pred
    tp = (gY * pY).sum(0).astype(float); fp = ((1 - gY) * pY).sum(0); fn = (gY * (1 - pY)).sum(0)
    P = tp / np.maximum(tp + fp, 1); R = tp / np.maximum(tp + fn, 1); F = np.where(P + R > 0, 2 * P * R / np.maximum(P + R, 1e-12), 0.0)
    return pd.DataFrame({"P": P, "R": R, "F1": F, "support": gY.sum(0).astype(int)}, index=LABELS)

# ---- bootstrap (paired: the same resampled indices are used for every model)
def make_boot_indices(n, B, rng, groups=None):
    if groups is None: return [rng.integers(0, n, n) for _ in range(B)]
    groups = np.asarray(groups); ug = np.unique(groups); members = [np.where(groups == g)[0] for g in ug]
    return [np.concatenate([members[i] for i in rng.integers(0, len(ug), len(ug))]) for _ in range(B)]

def boot_metrics(gold, pred, idxs, keys=("score", "cls_micro_F1", "cls_macro_F1", "ident_FR_F1", "ident_NFR_F1")):
    rows = [metrics_np(gold, pred, ix) for ix in idxs]
    return {k: np.array([r[k] for r in rows]) for k in keys}

def ci95(a): lo, hi = np.percentile(a, [2.5, 97.5]); return float(lo), float(hi)
def fmt_ci(pt, a): lo, hi = ci95(a); return f"{pt:.3f} [{lo:.3f}, {hi:.3f}]"
def boot_pvalue(d): return float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order): run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

def noninferiority_table(names, ref, point, boots, sizes, delta, alpha=0.05):
    rows = []
    for n in names:
        lo = float(np.percentile(boots[n] - boots[ref], 100 * alpha))
        rows.append(dict(name=n, size_MB=sizes[n], score=point[n], diff_vs_ref=point[n] - point[ref], lower_bound=lo, noninferior=bool(lo > -delta)))
    return pd.DataFrame(rows)
def choose_smallest(tab):
    ok = tab[tab.noninferior]
    return None if len(ok) == 0 else ok.sort_values(["size_MB", "score"], ascending=[True, False]).iloc[0]["name"]

# ---- project-wise folds + rare-class oversampling
def make_folds(d, n_folds, seed):
    """Whole projects -> folds, balanced by number of requirements (greedy)."""
    rng = np.random.default_rng(seed); sizes = d.project.value_counts()
    projs = list(sizes.index); rng.shuffle(projs); projs = sorted(projs, key=lambda p: -sizes[p])
    fold_of, load = {}, [0] * n_folds
    for p in projs:
        k = int(np.argmin(load)); fold_of[p] = k; load[k] += int(sizes[p])
    return fold_of

def oversample_rare(d, labels, min_count, seed):
    rng = np.random.default_rng(seed); extra = []
    for l in labels:
        pos = d[d[l] == 1]; n = len(pos)
        if 0 < n < min_count: extra.append(pos.sample(min_count - n, replace=True, random_state=int(rng.integers(1 << 30))))
    return d if not extra else pd.concat([d] + extra, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

# ---------------------------------------------------------------- v3 additions
def make_target_noexpl(row):
    """Target WITHOUT explanation (for the explanation-free ablation); parse_output() handles the missing explanation."""
    return pred_string(row["is_functional"], row["is_quality"], [l for l in LABELS if row[l] == 1], "").split(" | explanation:")[0]

# evidence grounding: phrases the explanation puts in double quotes should literally occur in the requirement
_QUOTE_RE = re.compile(r'["“]([^"”]{2,80})["”]')
def _norm_txt(s): return " ".join(re.sub(r"[^a-z0-9 ]", " ", str(s).lower()).split())
def evidence_counts(expl, req):
    """(number of quoted spans in the explanation, number of them found in the requirement)"""
    spans = [x for x in (_norm_txt(s) for s in _QUOTE_RE.findall(expl or "")) if x]
    rq = _norm_txt(req)
    return len(spans), sum(1 for s in spans if s in rq)


In [6]:
from sklearn.model_selection import GroupShuffleSplit
assert df.project.nunique() >= 5, "need at least 5 projects for a project-wise train/validation/test split"
_tr, _te = next(GroupShuffleSplit(n_splits=1, test_size=SPLIT["TEST_FRAC"], random_state=SEED).split(df, groups=df.project))
df["fold"] = df.project.map(make_folds(df, 5, SEED))  # 5 project-wise outer folds (replaces single GroupShuffleSplit)
_unused = 1                                   # 1 = train+validation pool (validation projects are carved out later, project-wise)
df["leak_fold"] = df.project.map(make_folds(df, LEAK_FOLDS, SEED))   # only for the TF-IDF leakage study
_pool = df[df.fold != 0]
_a, _b = next(GroupShuffleSplit(n_splits=1, test_size=SPLIT["VAL_FRAC"], random_state=SEED).split(_pool, groups=_pool.project))
split_tab = pd.DataFrame({
    "train":      [_pool.iloc[_a].project.nunique(), len(_pool.iloc[_a])],
    "validation": [_pool.iloc[_b].project.nunique(), len(_pool.iloc[_b])],
    "test":       [df[df.fold == 0].project.nunique(), int((df.fold == 0).sum())]}, index=["projects", "requirements"])
display(split_tab)
print("Single project-wise split: a project is never in more than one of train / validation / test. (Table is a preview; run_fold builds the exact validation split.)")


,train,validation,test
projects,9,3,3
requirements,320,175,127


Single project-wise split: a project is never in more than one of train / validation / test. (Table is a preview; run_fold builds the exact validation split.)


## 3. Explanation targets (Groq, free tier) — fails loudly instead of silently using templates

Gold labels are given to the LLM **only** to write the *target* explanation; the fine-tuned models never see labels or explanations as input. Requirements are sent in **batches of 8** with a shared rate limiter, and every finished batch is cached, so a disconnect or daily-quota stop loses nothing. Put the key in Colab **Secrets** (name `GROQ_API_KEY`, free at console.groq.com) or the environment. **Never paste a key into a cell.** Groq is the only external API in this notebook; the optional LLM judge in section 7 also runs on Groq (a different Groq model than the writer).


In [7]:
import os, re, json, time, hashlib, threading, subprocess, sys
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
_missing = [n for n in ("WORK_DIR", "df", "LABELS") if n not in globals()]
if _missing:
    raise RuntimeError(f"Not defined yet: {_missing}. The Colab session was restarted (or earlier cells were skipped). "
                       "Click this cell, choose Runtime -> Run before, then run this cell again.")
try: import groq
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "groq"]); import groq
from groq import Groq

# ------------------------------------------------------------------ Groq settings (free tier: ~30 req/min, ~8k tokens/min, ~200k tokens/day per model)
GROQ_MODEL = None                                   # None = first available model from the list below; or force e.g. "openai/gpt-oss-20b"
GROQ_MODEL_PREFS = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "llama-3.3-70b-versatile", "llama-3.1-8b-instant"]
BATCH_SIZE = 8                                      # requirements per API call (the free-tier bottleneck is tokens/min, so batching ~2x the speed)
WORKERS = 4                                         # parallel calls; a shared limiter keeps them inside the per-minute quota
# (Developer tier has ~10x the limits: raise TPM_BUDGET to ~60000 and RPM_BUDGET to ~250 there.)

BATCH_HEADER = """You are a requirements engineering expert. Below are numbered software requirements. For each one you are given its type and its non-functional categories.
For EACH requirement write 2-3 concise sentences (at most 60 words) explaining why it belongs to exactly these type(s) and categories.
Name every listed category explicitly, using exactly the category name as written, and point to the specific words or phrases in that requirement that justify each label.
Do not mention any category that is not listed for that requirement. If no categories are listed, say that the requirement is purely functional and why.
Treat every requirement independently.
Output format: one entry per requirement, each starting on a new line with its number in square brackets, e.g. "[3] This requirement is ...". No other text and no markdown."""

def gold_summary(row):
    kinds = [k for k, f in (("functional", row["is_functional"]), ("non-functional (quality)", row["is_quality"])) if f]
    return kinds, [l for l in LABELS if row[l] == 1]
def template_explanation(row):
    kinds, cats = gold_summary(row)
    s = ("This requirement is " + " and ".join(kinds) + ".") if kinds else "This requirement has no clear type."
    return s + (" It addresses " + ", ".join(cats) + "." if cats else "")

def _secret(name, *alts):
    for n in (name, *alts):
        k = os.environ.get(n)
        if k: return k
    try:
        from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret(name)
    except Exception: pass
    try:
        from google.colab import userdata; return userdata.get(name)
    except Exception: return None

# ------------------------------------------------------------------ start-up checks
oai, WRITER_MODEL, EXPL_VALID = None, None, True
JUDGE_MODEL_USED = None
JUDGE_MODEL_PREFS = ["llama-3.3-70b-versatile", "openai/gpt-oss-120b", "openai/gpt-oss-20b", "llama-3.1-8b-instant"]
GROQ_KEY = _secret("GROQ_API_KEY")                  # the NAME of the secret, never the key itself
if not GROQ_KEY:                                     # fallback: hidden prompt, so the key is never saved in the notebook
    from getpass import getpass
    GROQ_KEY = getpass("Enter GROQ_API_KEY: ").strip() or None
    if GROQ_KEY: os.environ["GROQ_API_KEY"] = GROQ_KEY
print("Groq key loaded:", (GROQ_KEY[:4] + "..." + GROQ_KEY[-4:]) if GROQ_KEY else "NO")
if USE_GROQ and GROQ_KEY:                           # USE_GROQ = "use the Groq LLM to write the explanation targets"
    oai = Groq(api_key=GROQ_KEY, max_retries=0, timeout=90)
    try: available = {m.id for m in oai.models.list().data}
    except Exception as e: raise RuntimeError(f"Could not reach Groq ({type(e).__name__}: {str(e)[:200]}). Check GROQ_API_KEY and internet access.")
    WRITER_MODEL = GROQ_MODEL or next((m for m in GROQ_MODEL_PREFS if m in available), None)
    if WRITER_MODEL is None or WRITER_MODEL not in available:
        raise RuntimeError(f"Groq model '{WRITER_MODEL}' not available for this key. Available text models: {sorted(available)}. Set GROQ_MODEL.")
    if RUN_LLM_JUDGE:                                # judge = a DIFFERENT Groq model than the writer whenever one is available
        if JUDGE_MODEL and JUDGE_MODEL not in available:
            raise RuntimeError(f"Judge model '{JUDGE_MODEL}' not available for this Groq key. Available: {sorted(available)}.")
        JUDGE_MODEL_USED = JUDGE_MODEL or next((m for m in JUDGE_MODEL_PREFS if m in available and m != WRITER_MODEL), None)
        if JUDGE_MODEL_USED is None:
            JUDGE_MODEL_USED = WRITER_MODEL
            print("WARNING: no second Groq model available -> the judge is the same model as the writer (weaker evidence).")
elif ALLOW_TEMPLATE_EXPLANATIONS:
    EXPL_VALID = False
    print("#" * 78, "\nTEMPLATE explanations in use -> ALL explanation metrics are disabled and must NOT be reported.\n" + "#" * 78)
else:
    raise RuntimeError("No Groq key found. Create a free key at console.groq.com, add it to Kaggle Secrets (or Colab Secrets) as GROQ_API_KEY, "
                       "or set ALLOW_TEMPLATE_EXPLANATIONS=True (explanation results will then be invalid).")

PROMPT_SHA = hashlib.sha256(((BATCH_HEADER + str(WRITER_MODEL)).encode())).hexdigest()[:8]
EXPL_CACHE = os.path.join(WORK_DIR, f"explanations_groq_{PROMPT_SHA}.json")
explanations = json.load(open(EXPL_CACHE)) if os.path.exists(EXPL_CACHE) else {}

# ------------------------------------------------------------------ rate limiter (sliding 60 s window over tokens and requests)
class Limiter:
    def __init__(s, tpm, rpm): s.tpm, s.rpm, s.lock, s.ev = tpm, rpm, threading.Lock(), deque()
    def acquire(s, est):
        while True:
            with s.lock:
                now = time.time()
                while s.ev and now - s.ev[0][0] > 60: s.ev.popleft()
                used = sum(x[1] for x in s.ev)
                if (not s.ev or used + est <= s.tpm) and len(s.ev) < s.rpm:
                    rec = [now, est]; s.ev.append(rec); return rec
                wait = (s.ev[0][0] + 60 - now) if s.ev else 1.0
            time.sleep(max(wait, 0.2) + 0.05)
    def settle(s, rec, actual):
        with s.lock: rec[1] = actual
TPM_BUDGET = 6500 if (WRITER_MODEL or "").startswith("openai/gpt-oss") else 5000
RPM_BUDGET = 25
LIM, STOP, SAVE_LOCK, _state = Limiter(TPM_BUDGET, RPM_BUDGET), threading.Event(), threading.Lock(), dict(reasoning_param=True)
class DailyQuota(RuntimeError): pass

def _chat(prompt, max_out, est_tokens, retries=6, model=None):
    model = model or WRITER_MODEL
    last = None
    for a in range(retries):
        if STOP.is_set(): raise DailyQuota("stopped: daily quota reached")
        rec = LIM.acquire(est_tokens)
        try:
            kw = dict(model=model, temperature=0, max_completion_tokens=max_out, messages=[{"role": "user", "content": prompt}])
            if model.startswith("openai/gpt-oss") and _state["reasoning_param"]:
                kw["extra_body"] = {"reasoning_effort": "low"}          # short hidden reasoning = faster + fewer tokens
            r = oai.chat.completions.create(**kw)
            u = getattr(r, "usage", None); LIM.settle(rec, getattr(u, "total_tokens", None) or est_tokens)
            return r.choices[0].message.content or "", r.choices[0].finish_reason
        except groq.RateLimitError as e:
            last = e
            if re.search(r"per day|\(TPD\)|\(RPD\)", str(e)): raise DailyQuota(str(e)[:300])
            try: ra = float(e.response.headers.get("retry-after"))
            except Exception: ra = None
            time.sleep((ra if ra else 2 ** a) + 1)
        except groq.BadRequestError as e:
            LIM.settle(rec, 0)
            if "reasoning" in str(e).lower() and _state["reasoning_param"]: _state["reasoning_param"] = False; continue
            raise
        except Exception as e:                                           # connection errors / 5xx
            last = e; LIM.settle(rec, 0); time.sleep(2 ** a)
    raise RuntimeError(f"no response after {retries} tries: {str(last)[:200]}")

def _item_line(k, row):
    kinds, cats = gold_summary(row)
    return f'[{k}] Requirement: "{" ".join(str(row["text"]).split())}" | Type: {" and ".join(kinds) or "unclear"} | Non-functional categories: {", ".join(cats) or "none (purely functional)"}'

def _parse(out, n):
    res = {}
    for m in re.finditer(r"^\s*\[(\d+)\]\s*(.*?)(?=^\s*\[\d+\]|\Z)", out, re.S | re.M):
        k, t = int(m.group(1)), " ".join(m.group(2).split())            # single line: '|' parsing stays unambiguous
        if 1 <= k <= n and t and k not in res: res[k] = t
    return res

def _do_batch(rows):
    n = len(rows)
    prompt = BATCH_HEADER + "\n\n" + "\n".join(_item_line(i + 1, r) for i, r in enumerate(rows))
    out, fin = _chat(prompt, max_out=110 * n + 500, est_tokens=len(prompt) / 3.5 + 95 * n + 150)
    got = _parse(out, n)
    if fin == "length" and got: got.pop(max(got))                       # last entry may be cut off
    return {rows[k - 1]["text"]: v for k, v in got.items()}

def _save():
    json.dump(explanations, open(EXPL_CACHE + ".tmp", "w"), indent=1); os.replace(EXPL_CACHE + ".tmp", EXPL_CACHE)

def _run(todo, bs):
    chunks, errs, quota = [todo[i:i + bs] for i in range(0, len(todo), bs)], [], None
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futs = [ex.submit(_do_batch, c) for c in chunks]
        for f in tqdm(as_completed(futs), total=len(futs), desc=f"Groq explanations ({WRITER_MODEL}, batch={bs})"):
            try: res = f.result()
            except DailyQuota as e: STOP.set(); quota = quota or str(e); continue
            except Exception as e: errs.append(str(e)[:200]); continue
            with SAVE_LOCK: explanations.update(res); _save()           # progress survives a disconnect
    if quota: raise RuntimeError(f"Groq daily quota for '{WRITER_MODEL}' is used up ({len(explanations)}/{len(df)} explanations cached). "
                                 f"Re-run this cell after the daily reset (UTC midnight) and it continues where it stopped. Detail: {quota}")
    return errs

if EXPL_VALID:
    errs = []
    for bs in (BATCH_SIZE, 2, 1):                                        # items the model skipped/garbled are retried in smaller batches
        todo = list({r["text"]: r for _, r in df.iterrows() if r["text"] not in explanations}.values())
        if not todo: break
        errs += _run(todo, bs)
    left = [t for t in df.text if t not in explanations]
    if left: raise RuntimeError(f"{len(left)} explanations still missing (first error: {errs[:1]}). Re-run this cell: finished ones are cached.")
    df["explanation"] = [explanations[t] for t in df.text]
else:
    df["explanation"] = [template_explanation(r) for _, r in df.iterrows()]

tcons = np.mean([label_consistency(r["explanation"], [l for l in LABELS if r[l] == 1]) for _, r in df.iterrows()])
print(f"Explanation targets: {len(df)} | label-consistent with gold labels: {tcons:.1%} | model: {WRITER_MODEL if EXPL_VALID else 'TEMPLATE'}")
if EXPL_VALID and tcons < 0.90: print("WARNING: <90% of targets name exactly the gold categories; inspect df.explanation before training.")
print("Example:", df.loc[0, "text"], "->", df.loc[0, "explanation"])

Enter GROQ_API_KEY:  ········


Groq key loaded: gsk_...ShNf


Groq explanations (openai/gpt-oss-120b, batch=8):   0%|          | 0/78 [00:00<?, ?it/s]

Explanation targets: 622 | label-consistent with gold labels: 91.3% | model: openai/gpt-oss-120b
Example: The system shall refresh the display every 60 seconds. -> This requirement is both functional and non‑functional because it specifies a system behavior (“shall refresh the display”) and also a quality attribute. The phrase “every 60 seconds” defines a performance constraint, so it belongs to the Performance category.


In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
import torch.nn.functional as F

# --- explanation quality gate: ~12% of Groq targets named categories that differ from the gold labels.
# Inconsistent targets would teach the model to contradict its own predicted categories, so they are replaced by the
# gold-derived template (consistent by construction). The flag is kept so explanation metrics can exclude them.
df["expl_llm_ok"] = [label_consistency(r["explanation"], [l for l in LABELS if r[l] == 1]) for _, r in df.iterrows()]
df["explanation_raw"] = df["explanation"]
df.loc[~df.expl_llm_ok, "explanation"] = [template_explanation(r) for _, r in df[~df.expl_llm_ok].iterrows()]
print(f"Explanation gate: {int((~df.expl_llm_ok).sum())}/{len(df)} targets ({(~df.expl_llm_ok).mean():.1%}) replaced by template; "
      f"consistency now {np.mean([label_consistency(r['explanation'], [l for l in LABELS if r[l]==1]) for _, r in df.iterrows()]):.1%}")
assert df.apply(lambda r: label_consistency(r["explanation"], [l for l in LABELS if r[l] == 1]), axis=1).all()
df["src"] = df.text.map(make_input); df["tgt"] = df.apply(make_target, axis=1); df["tgt_ne"] = df.apply(make_target_noexpl, axis=1)
_g = score_raw(df.tgt.tolist(), df)                       # pipeline sanity check: gold targets must score 1.0
assert _g["cls_micro_F1"] == 1.0 and _g["score"] == 1.0 and _g["format_ok"] == 1.0, _g
_g2 = score_raw(df.tgt_ne.tolist(), df)                   # explanation-free targets must also parse and score 1.0
assert _g2["score"] == 1.0 and _g2["format_ok"] == 1.0, _g2

# All T5 checkpoints used here share one 32k SentencePiece vocabulary (needed for token-level distillation).
tok = AutoTokenizer.from_pretrained(BASELINE_NAME)
tl = np.array([len(x) for x in tok(df.tgt.tolist())["input_ids"]])
print(f"Target length (tokens): median {np.median(tl):.0f}, 99th pct {np.percentile(tl, 99):.0f}, max {tl.max()} | MAX_OUT_LEN={MAX_OUT_LEN}")
assert (tl > MAX_OUT_LEN).mean() < 0.01, "more than 1% of targets would be truncated; raise MAX_OUT_LEN"

Explanation gate: 54/622 targets (8.7%) replaced by template; consistency now 100.0%


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Target length (tokens): median 65, 99th pct 112, max 130 | MAX_OUT_LEN=192


## 4. Models: training helpers (KD), quantization, external baselines

In [9]:
MODELS_DIR = os.path.join(WORK_DIR, "models"); os.makedirs(MODELS_DIR, exist_ok=True)
def free():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def tokenize_df(d, tgt_col="tgt"):
    def f(b):
        x = tok(b["src"], max_length=MAX_IN_LEN, truncation=True)
        x["labels"] = tok(text_target=b["tgt"], max_length=MAX_OUT_LEN, truncation=True)["input_ids"]
        return x
    return Dataset.from_pandas(d[["src", tgt_col]].rename(columns={tgt_col: "tgt"}).reset_index(drop=True)).map(f, batched=True, remove_columns=["src", "tgt"])

def make_compute_metrics(val_df):
    def compute_metrics(ev):                    # runs on the fold's VALIDATION projects after every epoch (greedy decoding)
        preds = ev.predictions[0] if isinstance(ev.predictions, tuple) else ev.predictions
        preds = np.where(preds < 0, tok.pad_token_id, preds)
        m = score_raw(tok.batch_decode(preds, skip_special_tokens=True), val_df)
        return {"score": m["score"], "micro_f1": m["cls_micro_F1"], "macro_f1": m["cls_macro_F1"]}
    return compute_metrics

class KDTrainer(Seq2SeqTrainer):
    """Seq2SeqTrainer + token-level distillation: loss = a*CE(gold) + (1-a)*T^2*KL(teacher||student). teacher=None -> plain fine-tuning."""
    def __init__(self, *args, teacher=None, alpha=0.5, temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher, self.alpha, self.T = teacher, alpha, temperature
        self.model_accepts_loss_kwargs = False
        if teacher is not None:
            self.teacher.to(self.args.device).eval()
            for p in self.teacher.parameters(): p.requires_grad_(False)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        out = model(**inputs); loss = out.loss
        if self.teacher is not None and model.training:
            with torch.no_grad(): t_logits = self.teacher(**inputs).logits
            mask = inputs["labels"].ne(-100)
            s = out.logits[mask] / self.T; t = t_logits[mask].to(s.dtype) / self.T
            kd = F.kl_div(F.log_softmax(s, -1), F.log_softmax(t, -1), log_target=True, reduction="batchmean") * self.T ** 2
            loss = self.alpha * loss + (1 - self.alpha) * kd
        return (loss, out) if return_outputs else loss

def train_seq2seq(model_name, tag, ds_train, ds_val, val_df, fold_dir, epochs, lr, bs=8, grad_acc=1, teacher=None, grad_ckpt=False, seed=SEED):
    """Fine-tune on the fold's TRAIN, keep the epoch with the best VALIDATION score, save to fold_dir/tag.
    Returns (path, best validation score, best epoch)."""
    set_seed(seed)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    if teacher is not None: assert model.config.vocab_size == teacher.config.vocab_size, "teacher/student vocabularies differ"
    steps = math.ceil(len(ds_train) / (bs * grad_acc)) * epochs
    out_ckpt = os.path.join(fold_dir, "ckpt_" + tag)
    args = Seq2SeqTrainingArguments(
        output_dir=out_ckpt, num_train_epochs=epochs, learning_rate=lr, per_device_train_batch_size=bs, per_device_eval_batch_size=16,
        gradient_accumulation_steps=grad_acc, weight_decay=0.01, warmup_steps=max(1, math.ceil(0.05 * steps)), optim="adafactor",
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, save_only_model=True, load_best_model_at_end=True,
        metric_for_best_model="score", greater_is_better=True, predict_with_generate=True, generation_max_length=MAX_OUT_LEN,
        generation_num_beams=1, gradient_checkpointing=grad_ckpt, fp16=torch.cuda.is_available(),
        logging_steps=50, report_to="none", seed=seed)   # fp32 (T4: no bf16)
    trainer = KDTrainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                        data_collator=DataCollatorForSeq2Seq(tok, model=model), compute_metrics=make_compute_metrics(val_df),
                        processing_class=tok, teacher=teacher, alpha=KD_ALPHA, temperature=KD_T)
    trainer.train()
    best = float(trainer.state.best_metric); out_dir = os.path.join(fold_dir, tag)
    hist = [h for h in trainer.state.log_history if "eval_score" in h]
    best_epoch = float(max(hist, key=lambda h: h["eval_score"])["epoch"]) if hist else float(epochs)   # needed to refit without a validation set
    trainer.model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    shutil.rmtree(out_ckpt, ignore_errors=True); del trainer, model; free()
    print(f"==> {tag}: best validation score {best:.3f} at epoch {best_epoch:.0f}")
    return out_dir, best, best_epoch

def train_with_lr_search(model_name, tag, lrs, fold_dir, **kw):
    """Train once per learning rate, choose the best by VALIDATION score only, keep only that model under fold_dir/tag.
    A single-element lrs just trains once. Returns (path, best_score, best_epoch, chosen_lr, {lr: validation score})."""
    runs = {}
    for lr in lrs:
        t = tag if len(lrs) == 1 else f"{tag}__lr{lr:.0e}"
        runs[lr] = train_seq2seq(model_name, t, fold_dir=fold_dir, lr=lr, **kw)
    lr_best = max(runs, key=lambda l: runs[l][1])
    best_path = runs[lr_best][0]; final = os.path.join(fold_dir, tag)
    if best_path != final:
        shutil.rmtree(final, ignore_errors=True); os.rename(best_path, final)
        for l, (p, _, _) in runs.items():
            if p != best_path: shutil.rmtree(p, ignore_errors=True)
    if len(lrs) > 1: print(f"   lr search for {tag}: " + ", ".join(f"{l:.0e}->{runs[l][1]:.3f}" for l in runs) + f"  => {lr_best:.0e}")
    return final, runs[lr_best][1], runs[lr_best][2], lr_best, {l: v[1] for l, v in runs.items()}

def train_fixed(model_name, tag, ds_train, out_root, epochs, lr, bs=8, grad_acc=1, teacher=None, grad_ckpt=False, seed=SEED):
    """Train for a FIXED number of epochs with no validation set (used to refit on all projects). Saves to out_root/tag."""
    set_seed(seed); os.makedirs(out_root, exist_ok=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    steps = math.ceil(len(ds_train) / (bs * grad_acc)) * epochs
    args = Seq2SeqTrainingArguments(
        output_dir=os.path.join(out_root, "ckpt_" + tag), num_train_epochs=epochs, learning_rate=lr, per_device_train_batch_size=bs,
        gradient_accumulation_steps=grad_acc, weight_decay=0.01, warmup_steps=max(1, math.ceil(0.05 * steps)), optim="adafactor",
        eval_strategy="no", save_strategy="no", gradient_checkpointing=grad_ckpt, fp16=torch.cuda.is_available(),
        logging_steps=50, report_to="none", seed=seed)
    trainer = KDTrainer(model=model, args=args, train_dataset=ds_train, data_collator=DataCollatorForSeq2Seq(tok, model=model),
                        processing_class=tok, teacher=teacher, alpha=KD_ALPHA, temperature=KD_T)
    trainer.train(); out_dir = os.path.join(out_root, tag)
    trainer.model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    shutil.rmtree(os.path.join(out_root, "ckpt_" + tag), ignore_errors=True); del trainer, model; free()
    return out_dir


In [10]:
from torch.ao.quantization import quantize_dynamic, default_dynamic_qconfig, float_qparams_weight_only_qconfig

def count_params(m): return sum(p.numel() for p in m.parameters())
def _fp32(path): return AutoModelForSeq2SeqLM.from_pretrained(path, torch_dtype=torch.float32)

def int8_spec(m, mode):
    if mode == "safe":   # keep lm_head and the FFN output projection (T5 'wo', known activation outliers) in fp32
        return {n: default_dynamic_qconfig for n, mod in m.named_modules()
                if isinstance(mod, torch.nn.Linear) and not (n.endswith("lm_head") or n.endswith(".wo"))}
    spec = {torch.nn.Linear: default_dynamic_qconfig}
    if mode == "emb": spec[torch.nn.Embedding] = float_qparams_weight_only_qconfig
    return spec

def load_int8(path, mode="plain"):
    m = _fp32(path).eval(); q = quantize_dynamic(m, int8_spec(m, mode), dtype=torch.qint8)
    if mode == "emb": q.encoder.embed_tokens = q.shared; q.decoder.embed_tokens = q.shared   # T5 shares one embedding table
    return q
def load_nf4(path):
    from transformers import BitsAndBytesConfig
    qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float32)
    return AutoModelForSeq2SeqLM.from_pretrained(path, quantization_config=qc, device_map={"": 0}).eval()

LOADERS = {"fp32": lambda p: _fp32(p).to(DEVICE).eval(), "int8": lambda p: load_int8(p, "plain"),
           "int8-safe": lambda p: load_int8(p, "safe"), "int8+emb": lambda p: load_int8(p, "emb"), "nf4": load_nf4}

def size_mb(model, kind):
    """Serialized size in MB, measured the same way for every kind (bytes actually written to disk)."""
    tmp = os.path.join(WORK_DIR, "_size_tmp")
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    try:
        if kind == "nf4":
            try: model.save_pretrained(tmp, safe_serialization=True)
            except Exception: return model.get_memory_footprint() / 1e6       # fallback (flagged by being slightly different)
        else: torch.save(model.state_dict(), os.path.join(tmp, "m.pt"))
        return sum(os.path.getsize(os.path.join(tmp, f)) for f in os.listdir(tmp)) / 1e6
    finally: shutil.rmtree(tmp, ignore_errors=True)

def generate_seq2seq(model, tokenizer, srcs, bs=16, max_new_tokens=MAX_OUT_LEN, beams=None, progress=False):
    beams = beams or EVAL_BEAMS; model.eval(); outs = []; dev = model.device
    for i in (tqdm(range(0, len(srcs), bs), desc="generate", leave=False) if progress else range(0, len(srcs), bs)):
        enc = tokenizer(srcs[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=MAX_IN_LEN).to(dev)
        with torch.no_grad(): g = model.generate(**enc, max_new_tokens=max_new_tokens, max_length=None, num_beams=beams)
        outs += tokenizer.batch_decode(g, skip_special_tokens=True)
    return outs

In [11]:
# ============================ external baselines (so the paper can show the LLM approach is worth its cost)
def tfidf_lr_predict(train_df, val_df, test_df, seed=0):
    from sklearn.linear_model import LogisticRegression
    cols = ["is_functional", "is_quality"] + LABELS
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train_df.text)
    Xtr, Xva, Xte = [vec.transform(d.text) for d in (train_df, val_df, test_df)]
    def fit(C):
        ms = []
        for c in cols:
            y = train_df[c].values
            ms.append(LogisticRegression(C=C, class_weight="balanced", max_iter=2000).fit(Xtr, y) if 0 < y.sum() < len(y) else None)
        return ms
    def predict(ms, X):
        return np.column_stack([(m.predict_proba(X)[:, 1] > 0.5).astype(int) if m is not None else np.zeros(X.shape[0], int) for m in ms])
    best = None
    for C in (0.3, 1, 3, 10, 30):                        # C tuned on the fold's validation projects only
        ms = fit(C); Pv = predict(ms, Xva)
        s = metrics_np(gold_arrays(val_df), (Pv[:, 0], Pv[:, 1], Pv[:, 2:]))["score"]
        if best is None or s > best[0]: best = (s, C, ms)
    ms = best[2]; Pv, Pt = predict(ms, Xva), predict(ms, Xte)
    raw = lambda P: arrays_to_raw(P[:, 0], P[:, 1], P[:, 2:])
    nparam = sum(m.coef_.size for m in ms if m is not None)
    return raw(Pv), raw(Pt), len(pickle.dumps(ms)) / 1e6, nparam / 1e6, dict(C=best[1])

def encoder_predict(train_df, val_df, test_df, seed=0):
    from transformers import AutoTokenizer as _AT, AutoModelForSequenceClassification as _AM
    cols = ["is_functional", "is_quality"] + LABELS; set_seed(seed)
    etok = _AT.from_pretrained(ENCODER_NAME)
    model = _AM.from_pretrained(ENCODER_NAME, num_labels=len(cols), problem_type="multi_label_classification").to(DEVICE)
    Ytr = torch.tensor(train_df[cols].values, dtype=torch.float32); texts = train_df.text.tolist()
    steps = math.ceil(len(texts) / ENC_BS) * ENC_EPOCHS; warm = max(1, int(0.1 * steps))
    pos = Ytr.sum(0); pos_w = ((len(Ytr) - pos) / pos.clamp(min=1)).clamp(max=10.0).to(DEVICE)   # counter label imbalance
    bce = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=ENC_LR, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, (steps - s) / steps))
    def probs(d):
        model.eval(); out = []; tx = d.text.tolist()
        for i in range(0, len(tx), 64):
            b = etok(tx[i:i + 64], truncation=True, max_length=MAX_IN_LEN, padding=True, return_tensors="pt").to(DEVICE)
            with torch.no_grad(): out.append(torch.sigmoid(model(**b).logits).float().cpu().numpy())
        return np.concatenate(out)
    to_arr = lambda p, t: ((p >= t).astype(int)[:, 0], (p >= t).astype(int)[:, 1], (p >= t).astype(int)[:, 2:])   # t: scalar or per-column vector
    best = (-1.0, None); g = torch.Generator().manual_seed(seed)
    for ep in range(ENC_EPOCHS):
        model.train(); perm = torch.randperm(len(texts), generator=g).tolist()
        for i in range(0, len(perm), ENC_BS):
            ix = perm[i:i + ENC_BS]
            b = etok([texts[j] for j in ix], truncation=True, max_length=MAX_IN_LEN, padding=True, return_tensors="pt").to(DEVICE)
            loss = bce(model(**b).logits, Ytr[ix].to(DEVICE))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad()
        s = metrics_np(gold_arrays(val_df), to_arr(probs(val_df), 0.5))["score"]
        if s > best[0]: best = (s, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
    model.load_state_dict(best[1]); pv, pt = probs(val_df), probs(test_df)
    grid = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)
    thr = np.full(pv.shape[1], max(grid, key=lambda t: metrics_np(gold_arrays(val_df), to_arr(pv, t))["score"]))   # global start
    for j in range(pv.shape[1]):                                  # coordinate ascent per label, VALIDATION only
        best_t, best_s = thr[j], -1.0
        for t in grid:
            th = thr.copy(); th[j] = t; sc = metrics_np(gold_arrays(val_df), to_arr(pv, th))["score"]
            if sc > best_s: best_t, best_s = t, sc
        thr[j] = best_t
    raw = lambda p: arrays_to_raw(*to_arr(p, thr)); npar = count_params(model)
    del model, opt; free()
    return raw(pv), raw(pt), npar * 4 / 1e6, npar / 1e6, dict(threshold=[float(x) for x in thr])

### Compute-aware execution notes

This run deliberately trains only one student architecture (T5-small), with one learning rate per candidate. The T5-large teacher is trained once and used for KD. The no-KD student is retained as the key distillation ablation. Only INT8-safe is run by default; NF4 can be added later after the first run if your GPU/time budget allows.

Mixed precision (`fp16`) is enabled automatically only when CUDA is available, and evaluation batch size is reduced to 16. The model-selection logic remains validation-based.


## 5. Train on a single project-wise train / validation / test split (resumable)

One single split (no folds, no cross-validation): hold out whole projects as **test**; hold out whole projects of the remainder as **validation** (checkpoint choice, teacher choice, tuning of the classical/BERT baselines); train baseline, teacher, KD students and no-KD students; quantize; generate validation **and** test predictions for every candidate. Test predictions are stored but **never used for any choice** (selection happens later, in section 6, from validation predictions only).

Per-fold results are saved to `nice_work_v3/fold{k}_s{seed}/fold_result.pkl`; after a disconnect just re-run this cell.

**v3:** the loop runs with one training seed (`TRAIN_SEEDS = (42,)`). The learning-rate search (validation projects only) runs for each fold and its choice is cached in `lr_choice_fold{k}.json`.

In [12]:
def role_of(tag, kind):
    if tag == "teacher": return "teacher"
    if tag.endswith(("_noKD", "_noExpl")): return "ablation"
    return "reference" if (tag == "baseline" and kind == "fp32") else "compressed"

def run_fold(k, tseed):
    """One outer fold, one TRAINING seed. Splits depend only on (SEED, k) -> identical val/test sets for every training seed."""
    fdir = os.path.join(WORK_DIR, f"fold{k}_s{tseed}"); os.makedirs(fdir, exist_ok=True)
    res_path = os.path.join(fdir, "fold_result.pkl")
    if os.path.exists(res_path): print(f"[fold {k} seed {tseed}] done already -> skip (delete {res_path} to redo)"); return
    t_start = time.time(); tr_seed = tseed + k; set_seed(tr_seed)
    lr_cache_path = os.path.join(WORK_DIR, f"lr_choice_fold{k}.json")     # learning rates are searched once per fold, then reused by later seeds
    lr_choice = json.load(open(lr_cache_path)) if os.path.exists(lr_cache_path) else {}

    # ---------------- project-wise split: test projects | validation projects | training projects
    test_df = df[df.fold == k].reset_index(drop=True); rest = df[df.fold != k].reset_index(drop=True)
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=INNER_VAL_GROUP_FRAC, random_state=SEED + k).split(rest, groups=rest.project))
    train_df, val_df = rest.iloc[a].reset_index(drop=True), rest.iloc[b].reset_index(drop=True)

    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train_df.text)        # fitted on TRAIN only
    Xtr = vec.transform(train_df.text)
    nd = lambda d: cosine_similarity(vec.transform(d.text), Xtr).max(axis=1) >= NEAR_DUP_THRESH
    nd_val, nd_test = nd(val_df), nd(test_df)
    val_df, test_df = val_df[~nd_val].reset_index(drop=True), test_df[~nd_test].reset_index(drop=True)

    assert not set(train_df.project) & set(test_df.project), "LEAKAGE: project in train and test"
    assert not set(train_df.project) & set(val_df.project), "LEAKAGE: project in train and validation"
    assert not set(val_df.project) & set(test_df.project), "LEAKAGE: project in validation and test"
    for (n1, d1), (n2, d2) in [(("train", train_df), ("val", val_df)), (("train", train_df), ("test", test_df)), (("val", val_df), ("test", test_df))]:
        assert not set(d1._key) & set(d2._key), f"LEAKAGE: identical requirement in {n1} and {n2}"
    leak = dict(fold=k, train_seed=tseed, n_train=len(train_df), n_val=len(val_df), n_test=len(test_df), near_dups_removed_val=int(nd_val.sum()),
                near_dups_removed_test=int(nd_test.sum()), near_dup_threshold=NEAR_DUP_THRESH, projects_train=sorted(map(str, set(train_df.project))),
                projects_val=sorted(map(str, set(val_df.project))), projects_test=sorted(map(str, set(test_df.project))),
                oversampling=OVERSAMPLE_RARE, test_used_for_selection=False, split="project-wise")
    print(f"[fold {k} seed {tseed}] train {len(train_df)} | val {len(val_df)} | test {len(test_df)} | near-dups removed val/test: {leak['near_dups_removed_val']}/{leak['near_dups_removed_test']}")

    train_fit = oversample_rare(train_df, LABELS, MIN_CLASS_COUNT, SEED + k) if OVERSAMPLE_RARE else train_df
    train_ds, val_ds = tokenize_df(train_fit), tokenize_df(val_df)
    train_ds_ne = tokenize_df(train_fit, "tgt_ne") if RUN_NOEXPL_ABLATION else None
    val_ds_ne = tokenize_df(val_df, "tgt_ne") if RUN_NOEXPL_ABLATION else None

    # ---------------- train: baseline, teacher, students (KD and no-KD), explanation-free baseline. LR chosen on VALIDATION only.
    trained, ckpt_best, best_ep, lr_used, sweep_log = {}, {}, {}, {}, []
    def fit(model_name, tag, lr_list, epochs, ds_tr, ds_va, **kw):
        lrs = [lr_choice[tag]] if tag in lr_choice else list(lr_list)
        path, best, ep, lr, sweep = train_with_lr_search(model_name, tag, lrs, fdir, ds_train=ds_tr, ds_val=ds_va, val_df=val_df, epochs=epochs, seed=tr_seed, **kw)
        trained[tag], ckpt_best[tag], best_ep[tag], lr_used[tag] = path, best, ep, lr
        for l, v in sweep.items(): sweep_log.append(dict(tag=tag, lr=l, val_score=v, seed=tseed, fold=k))
        if tag not in lr_choice: lr_choice[tag] = lr; json.dump(lr_choice, open(lr_cache_path, "w"))

    fit(BASELINE_NAME, "baseline", BASE_LR_GRID, BASE_EPOCHS, train_ds, val_ds, bs=8)
    teacher_tag = "baseline"
    if TEACHER_NAME != BASELINE_NAME and torch.cuda.is_available():
        try:
            fit(TEACHER_NAME, "teacher", [TEACHER_LR], TEACHER_EPOCHS, train_ds, val_ds, bs=4, grad_acc=2, grad_ckpt=True)   # teacher lr fixed (cost)
            if ckpt_best["teacher"] >= ckpt_best["baseline"]: teacher_tag = "teacher"     # decided on VALIDATION only
        except torch.cuda.OutOfMemoryError:
            free(); print("Teacher OOM -> baseline used as teacher")
    leak["teacher_used"] = teacher_tag
    if RUN_NOEXPL_ABLATION:
        try: fit(BASELINE_NAME, "baseline_noExpl", BASE_LR_GRID, BASE_EPOCHS, train_ds_ne, val_ds_ne, bs=8)
        except Exception as e: print(f"Skipping baseline_noExpl: {type(e).__name__}: {str(e)[:200]}"); free()
    teacher = _fp32(trained[teacher_tag]).to(DEVICE).eval()
    for name, lr0, ep in STUDENTS:
        tag = name.split("/")[-1]; ep = max(2, int(round(ep * CFG["STUDENT_EPOCH_SCALE"])))
        lrs = [lr0 * m for m in STUDENT_LR_MULTS] if TUNE_STUDENTS else [lr0]          # KD and no-KD students get the SAME tuning budget
        for variant, t in (("student_" + tag, teacher),) + ((("student_" + tag + "_noKD", None),) if TRY_NOKD_ABLATION else ()):
            try: fit(name, variant, lrs, ep, train_ds, val_ds, bs=16, teacher=t)
            except Exception as e: print(f"Skipping {variant}: {type(e).__name__}: {str(e)[:200]}"); free()
    del teacher; free()

    # ---------------- quantize + generate validation AND test predictions for every candidate
    cands, params_M = {}, {}
    def add_candidate(tag, kind):
        name = f"{tag} [{kind}]"; m = None
        try:
            m = LOADERS[kind](trained[tag])
            if kind == "fp32": params_M[tag] = count_params(m) / 1e6
            bs = 32 if str(m.device).startswith("cuda") else 8
            v = generate_seq2seq(m, tok, val_df.src.tolist(), bs=bs); t = generate_seq2seq(m, tok, test_df.src.tolist(), bs=bs)
            cands[name] = dict(name=name, tag=tag, kind=kind, role=role_of(tag, kind), params_M=params_M.get(tag, np.nan),
                               size_MB=size_mb(m, kind), val_raw=v, test_raw=t, ckpt_val=ckpt_best[tag], path=trained[tag],
                               best_epoch=best_ep[tag], lr=lr_used[tag])
            print(f"  {name:<46} {cands[name]['size_MB']:8.1f} MB   val score {score_raw(v, val_df)['score']:.3f}")
        except Exception as e: print(f"  {name} skipped: {type(e).__name__}: {str(e)[:200]}")
        finally: del m; free()
    for tag in trained:
        kinds = ("fp32",) if (tag == "teacher" or tag.endswith(("_noKD", "_noExpl"))) else ("fp32",) + tuple(QUANT_KINDS)
        for kind in kinds: add_candidate(tag, kind)

    # ---------------- external baselines
    ext = [("tfidf-logreg", lambda: tfidf_lr_predict(train_df, val_df, test_df))]
    if RUN_ENCODER_BASELINE: ext.append((ENCODER_NAME, lambda: encoder_predict(train_df, val_df, test_df, seed=tr_seed)))
    for nm, fn in ext:
        try:
            v, t, size, npar, info = fn()
            cands[nm] = dict(name=nm, tag=nm, kind="fp32", role="external", params_M=npar, size_MB=size, val_raw=v, test_raw=t,
                             ckpt_val=np.nan, path=None, info=info, best_epoch=np.nan, lr=np.nan)
            print(f"  {nm:<46} {size:8.1f} MB   val score {score_raw(v, val_df)['score']:.3f}")
        except Exception as e: print(f"  {nm} skipped: {type(e).__name__}: {str(e)[:200]}")
        finally: free()

    pickle.dump(dict(k=k, seed=tseed, train_df=train_df, val_df=val_df, test_df=test_df, cands=cands, leak=leak, seconds=time.time() - t_start,
                     lr_sweep=sweep_log, lr_used=lr_used, best_epoch=best_ep,
                     config=dict(PRESET=PRESET, STUDENTS=STUDENTS, QUANT_KINDS=QUANT_KINDS, EVAL_BEAMS=EVAL_BEAMS)), open(res_path, "wb"))
    # housekeeping: teacher, no-KD and no-explanation weights are not needed after evaluation
    for tag in list(trained):
        if (not KEEP_MODELS) or tag == "teacher" or tag.endswith(("_noKD", "_noExpl")): shutil.rmtree(trained[tag], ignore_errors=True)
    print(f"[fold {k} seed {tseed}] finished in {(time.time() - t_start) / 60:.1f} min")


In [13]:
# ===== Experiment plan: 5 project-wise folds x 3 seeds, run in STAGES (training is resumable; each fold/seed saves fold_result.pkl)
# Measured on your T4: ~43 min per fold-seed  ->  5 folds x 1 seed ~ 3.6 h ; x 3 seeds ~ 11 h (too long for one free session).
# The loop below is seed-major, so after stage 1 you already hold a complete 5-fold result set.
#   Stage 1: STAGE_SEEDS = (42,)        -> run, then analysis cells (set TRAIN_SEEDS accordingly)
#   Stage 2: STAGE_SEEDS = (42, 43)     -> only the new seed trains (finished folds are skipped)
#   Stage 3: STAGE_SEEDS = (42, 43, 44)
STAGE_SEEDS = (42,)
assert set(STAGE_SEEDS) <= set(CFG["TRAIN_SEEDS"])
TRAIN_SEEDS = list(STAGE_SEEDS)          # analysis cells use exactly the seeds that have finished
FOLDS = list(CFG["RUN_FOLDS"])
print("Will train folds", FOLDS, "for seeds", TRAIN_SEEDS, "| already finished:",
      [f"fold{k}_s{s}" for s in TRAIN_SEEDS for k in FOLDS if os.path.exists(os.path.join(WORK_DIR, f"fold{k}_s{s}", "fold_result.pkl"))])


Will train folds [0, 1, 2, 3, 4] for seeds [42] | already finished: []


In [14]:
FOLDS = CFG["RUN_FOLDS"] if CFG["RUN_FOLDS"] is not None else [0]
if PRESET == "quick": print("QUICK preset: smoke test only. Do NOT report these numbers.")
print(f"{len(TRAIN_SEEDS)} training seed(s) x {len(FOLDS)} split(s) | learning-rate grid: baseline {BASE_LR_GRID}, students x{STUDENT_LR_MULTS} "
      f"(searched on the first seed's run of each fold, reused by later seeds)")
for tseed in TRAIN_SEEDS:             # seed-major order: after the first seed you already have a complete (single-seed) result set
    for k in FOLDS: run_fold(k, tseed)


1 training seed(s) x 5 split(s) | learning-rate grid: baseline (0.0003,), students x(1.0,) (searched on the first seed's run of each fold, reused by later seeds)
[fold 0 seed 42] train 320 | val 175 | test 125 | near-dups removed val/test: 0/2


Map:   0%|          | 0/423 [00:00<?, ? examples/s]

Map:   0%|          | 0/175 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

... (747 outputs left out)

  student_flan-t5-small_noKD [fp32]                 307.9 MB   val score 0.573
  tfidf-logreg                                        0.5 MB   val score 0.638


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  bert-base-uncased                                 438.0 MB   val score 0.801
[fold 4 seed 42] finished in 43.4 min


## 6. Held-out test results with confidence intervals

Every requirement is predicted exactly once as part of a held-out project fold. CIs are **cluster-bootstrap 95% intervals (resampling whole projects)**, the conservative choice because requirements within a project are not independent. Differences between models use the *same* resampled projects (paired bootstrap); p-values are Holm-corrected over the comparisons with the baseline.

**v3:** one training seed (42) is used; the cluster bootstrap resamples projects, so the CIs reflect data sampling.

In [15]:
_missing = [n for n in ("TRAIN_SEEDS", "FOLDS", "WORK_DIR", "gold_arrays", "parse_output", "run_fold") if n not in globals()]
if _missing:
    raise RuntimeError(f"Not defined yet: {_missing}. The Colab session was restarted. Click this cell, then "
                       "Runtime -> Run before (training resumes from the saved checkpoints), then run this cell again.")
_todo = [f"fold{k}_s{s}" for s in TRAIN_SEEDS for k in FOLDS
         if not os.path.exists(os.path.join(WORK_DIR, f"fold{k}_s{s}", "fold_result.pkl"))]
if _todo:
    raise RuntimeError(f"Training is not finished for: {_todo}. Run the training cell (section 5) until it completes, then run this cell.")
SEEDS, S0 = list(TRAIN_SEEDS), TRAIN_SEEDS[0]
R = {(k, s): pickle.load(open(os.path.join(WORK_DIR, f"fold{k}_s{s}", "fold_result.pkl"), "rb")) for s in SEEDS for k in FOLDS}
for k in FOLDS:                                   # evaluation sets must be IDENTICAL across training seeds
    for s in SEEDS[1:]:
        assert R[(k, s)]["test_df"].text.tolist() == R[(k, S0)]["test_df"].text.tolist(), "test sets differ between seeds"
        assert R[(k, s)]["val_df"].text.tolist() == R[(k, S0)]["val_df"].text.tolist(), "validation sets differ between seeds"
REF = "baseline [fp32]"
all_names = list(R[(FOLDS[0], S0)]["cands"])
names = [n for n in all_names if all(n in r["cands"] for r in R.values())]
if set(all_names) - set(names): print("WARNING: excluded (missing in some folds/seeds):", sorted(set(all_names) - set(names)))
assert REF in names
names = [REF] + [n for n in names if n != REF]
ROLE = {n: R[(FOLDS[0], S0)]["cands"][n]["role"] for n in names}
SIZE = {n: float(np.mean([r["cands"][n]["size_MB"] for r in R.values()])) for n in names}
PARAMS = {n: float(np.nanmean([r["cands"][n]["params_M"] for r in R.values()])) for n in names}

val_all = pd.concat([R[(k, S0)]["val_df"].assign(outer=k) for k in FOLDS], ignore_index=True)
test_all = pd.concat([R[(k, S0)]["test_df"].assign(outer=k) for k in FOLDS], ignore_index=True)
gold_val, gold_test = gold_arrays(val_all), gold_arrays(test_all)

# ARRS[model][split][seed] = (fr, q, Y, format_ok, explanations, raw);  ARR[model][split] = first seed (qualitative analyses only)
ARRS = {}
for n in names:
    ARRS[n] = {"val": {}, "test": {}}
    for split, ref_df in (("val", val_all), ("test", test_all)):
        for s in SEEDS:
            raw = sum((R[(k, s)]["cands"][n][f"{split}_raw"] for k in FOLDS), [])
            assert len(raw) == len(ref_df), (n, split, s, len(raw), len(ref_df))
            parsed = [parse_output(r) for r in raw]
            ARRS[n][split][s] = parsed_to_arrays(parsed) + (np.array([p["ok"] for p in parsed]), [p["expl"] for p in parsed], raw)
ARR = {n: {sp: ARRS[n][sp][S0] for sp in ("val", "test")} for n in names}

KEYS = ("score", "cls_micro_F1", "cls_macro_F1", "ident_FR_F1", "ident_NFR_F1")
def seed_metrics(n, split, gold, idx=None):
    """Metrics averaged over the training seeds (same evaluation rows for every seed)."""
    ms = [metrics_np(gold, ARRS[n][split][s][:3], idx) for s in SEEDS]
    return {k: float(np.mean([m[k] for m in ms])) for k in ms[0]}
def seed_boot(n, split, gold, idxs, keys=KEYS):
    bs = [boot_metrics(gold, ARRS[n][split][s][:3], idxs, keys) for s in SEEDS]
    return {k: np.mean([b[k] for b in bs], axis=0) for k in keys}
def seed_sd(n, split="test", gold=None):
    gold = gold_test if gold is None else gold
    v = [metrics_np(gold, ARRS[n][split][s][:3])["score"] for s in SEEDS]
    return float(np.std(v, ddof=1)) if len(v) > 1 else float("nan")
fmt_ok = lambda n, split="test": float(np.mean([ARRS[n][split][s][3].mean() for s in SEEDS]))

rng = np.random.default_rng(SEED)
test_groups = test_all.project.values if BOOT_UNIT == "project" else None
IDX_TEST = make_boot_indices(len(test_all), BOOT_N, rng, test_groups)
PT = {n: seed_metrics(n, "test", gold_test) for n in names}
BT = {n: seed_boot(n, "test", gold_test, IDX_TEST) for n in names}

def paired(a, b, key="score"):
    d = BT[a][key] - BT[b][key]; lo, hi = ci95(d)
    return dict(diff=PT[a][key] - PT[b][key], lo=lo, hi=hi, p=boot_pvalue(d))

rows = []
for n in names:
    r = dict(model=n, role=ROLE[n], params_M=PARAMS[n], size_MB=SIZE[n], compression_x=SIZE[REF] / SIZE[n], format_ok=fmt_ok(n))
    for key, lab in (("score", "score"), ("cls_micro_F1", "micro-F1"), ("cls_macro_F1", "macro-F1"), ("ident_FR_F1", "FR-F1"), ("ident_NFR_F1", "NFR-F1")):
        r[lab] = fmt_ci(PT[n][key], BT[n][key])
    r["score_pt"] = PT[n]["score"]; r["seed_sd"] = seed_sd(n); r["exact_match_NFR"] = PT[n]["cls_exact_match_nfr"]; r["hamming"] = PT[n]["cls_hamming_loss"]
    if n != REF:
        pp = paired(n, REF); r["d_score_vs_base"] = f"{pp['diff']:+.3f} [{pp['lo']:+.3f}, {pp['hi']:+.3f}]"; r["_p"] = pp["p"]
    rows.append(r)
headline = pd.DataFrame(rows)
mask = headline["_p"].notna(); headline.loc[mask, "p_holm_vs_base"] = holm(headline.loc[mask, "_p"].values); headline = headline.drop(columns="_p")
headline = headline.sort_values("size_MB").reset_index(drop=True)
print(f"Held-out TEST results: {len(test_all)} requirements, {test_all.project.nunique()} projects, {len(FOLDS)} folds, "
      f"{len(SEEDS)} training seed(s) averaged (bootstrap unit: {BOOT_UNIT}, B={BOOT_N}; seed_sd = sd of the score across seeds)")
display(headline.drop(columns=["score_pt"]).round(3))
expl_tab = judge_tab = lat_tab = None


Held-out TEST results: 617 requirements, 15 projects, 5 folds, 1 training seed(s) averaged (bootstrap unit: project, B=1000; seed_sd = sd of the score across seeds)


,model,role,params_M,size_MB,compression_x,format_ok,score,micro-F1,macro-F1,FR-F1,NFR-F1,seed_sd,exact_match_NFR,hamming,d_score_vs_base,p_holm_vs_base
0,tfidf-logreg,external,0.059,0.475,2084.087,1.0,"0.529 [0.494, 0.562]","0.412 [0.340, 0.493]","0.340 [0.286, 0.390]","0.572 [0.485, 0.644]","0.791 [0.716, 0.865]",NaN,0.277,0.065,"-0.146 [-0.185, -0.114]",0.000
1,student_flan-t5-small [int8-safe],compressed,76.961,201.056,4.926,1.0,"0.535 [0.478, 0.580]","0.362 [0.286, 0.420]","0.327 [0.257, 0.376]","0.705 [0.607, 0.787]","0.745 [0.633, 0.816]",NaN,0.218,0.087,"-0.140 [-0.196, -0.084]",0.000
2,student_flan-t5-small [fp32],compressed,76.961,307.919,3.217,1.0,"0.547 [0.499, 0.598]","0.414 [0.354, 0.475]","0.370 [0.321, 0.416]","0.632 [0.519, 0.747]","0.772 [0.707, 0.833]",NaN,0.293,0.085,"-0.127 [-0.178, -0.072]",0.000
3,student_flan-t5-small_noKD [fp32],ablation,76.961,307.919,3.217,1.0,"0.561 [0.509, 0.607]","0.431 [0.387, 0.474]","0.393 [0.350, 0.430]","0.674 [0.573, 0.752]","0.747 [0.681, 0.818]",NaN,0.301,0.085,"-0.113 [-0.172, -0.057]",0.000
4,bert-base-uncased,external,109.492,437.969,2.261,1.0,"0.699 [0.664, 0.732]","0.581 [0.543, 0.631]","0.541 [0.495, 0.590]","0.795 [0.699, 0.859]","0.881 [0.834, 0.918]",NaN,0.319,0.061,"+0.025 [-0.012, +0.062]",0.376
5,baseline [int8-safe],compressed,247.578,509.263,1.945,1.0,"0.676 [0.649, 0.697]","0.545 [0.516, 0.576]","0.510 [0.460, 0.546]","0.786 [0.722, 0.832]","0.861 [0.806, 0.903]",NaN,0.327,0.067,"+0.001 [-0.010, +0.010]",0.856
6,baseline [fp32],reference,247.578,990.421,1.000,1.0,"0.674 [0.649, 0.698]","0.536 [0.505, 0.573]","0.513 [0.461, 0.546]","0.799 [0.733, 0.848]","0.849 [0.799, 0.890]",NaN,0.327,0.066,NaN,NaN
7,teacher [fp32],teacher,783.150,3132.818,0.316,1.0,"0.748 [0.713, 0.772]","0.625 [0.575, 0.675]","0.602 [0.552, 0.631]","0.870 [0.816, 0.905]","0.897 [0.855, 0.927]",NaN,0.426,0.055,"+0.074 [+0.033, +0.110]",0.000


In [16]:
# ---------- model selection: smallest candidate that is NON-INFERIOR to the baseline on VALIDATION data only (seed-averaged)
elig = [n for n in names if ROLE[n] == "compressed"]
cand_pool = [REF] + elig
rng_sel = np.random.default_rng(SEED + 1)

def select_on(mask):
    gv = tuple(a[mask] for a in gold_val)
    sub = {n: [tuple(a[mask] for a in ARRS[n]["val"][s][:3]) for s in SEEDS] for n in cand_pool}
    groups = val_all.project.values[mask] if BOOT_UNIT == "project" else None
    idx = make_boot_indices(int(mask.sum()), BOOT_N, rng_sel, groups)
    sc = lambda n, ix=None: float(np.mean([metrics_np(gv, p, ix)["score"] for p in sub[n]]))
    boots = {n: np.array([sc(n, ix) for ix in idx]) for n in cand_pool}
    point = {n: sc(n) for n in cand_pool}
    tab = noninferiority_table(elig, REF, point, boots, SIZE, NONINF_DELTA)
    ch = choose_smallest(tab); fallback = ch is None
    if fallback: ch = max(elig, key=lambda n: point[n])
    return ch, tab, fallback

# honest evaluation of the whole PROCEDURE: for outer fold j the choice may only use validation rows that do not come from
# fold j's test projects, so no label of fold j's test projects influences which model is picked for fold j.
sel = {}
for j in FOLDS:
    tp = set(R[(j, S0)]["test_df"].project)
    m = ((val_all.outer != j) & (~val_all.project.isin(tp))) | (val_all.outer == j)
    sel[j] = select_on(m.values)
print("Model chosen per outer fold:", {j: sel[j][0] for j in FOLDS})

DEPLOY, DEPLOY_TAB, DEPLOY_FALLBACK = select_on(np.ones(len(val_all), bool))
DEPLOY_TAG, DEPLOY_KIND = R[(FOLDS[0], S0)]["cands"][DEPLOY]["tag"], R[(FOLDS[0], S0)]["cands"][DEPLOY]["kind"]
print(f"\nDEPLOYMENT CHOICE (all validation data, seed-averaged; margin delta={NONINF_DELTA}): {DEPLOY}" + ("   [no candidate was non-inferior -> best-scoring compressed model]" if DEPLOY_FALLBACK else ""))
display(DEPLOY_TAB.sort_values("size_MB").round(3))

# procedure-level pooled test result (each fold's own choice), per seed, then averaged
def proc_arrays(s):
    pf, pq_, pP = (np.zeros(len(test_all), int), np.zeros(len(test_all), int), np.zeros((len(test_all), len(LABELS)), int))
    for j in FOLDS:
        rows_j = np.where(test_all.outer.values == j)[0]; a_ = ARRS[sel[j][0]]["test"][s]
        pf[rows_j], pq_[rows_j], pP[rows_j] = a_[0][rows_j], a_[1][rows_j], a_[2][rows_j]
    return pf, pq_, pP
PROC = {s: proc_arrays(s) for s in SEEDS}
_pm = [metrics_np(gold_test, PROC[s]) for s in SEEDS]
PROC_PT = {k: float(np.mean([m[k] for m in _pm])) for k in _pm[0]}
_pb = [boot_metrics(gold_test, PROC[s], IDX_TEST) for s in SEEDS]
PROC_BT = {k: np.mean([b[k] for b in _pb], axis=0) for k in _pb[0]}
d = PROC_BT["score"] - BT[REF]["score"]
print("\nProcedure-level (per-fold selected model) vs baseline on pooled test:")
print(f"  score {fmt_ci(PROC_PT['score'], PROC_BT['score'])} | macro-F1 {fmt_ci(PROC_PT['cls_macro_F1'], PROC_BT['cls_macro_F1'])} | "
      f"delta vs baseline {PROC_PT['score'] - PT[REF]['score']:+.3f} [{ci95(d)[0]:+.3f}, {ci95(d)[1]:+.3f}]")
pp = paired(DEPLOY, REF)
print(f"Deployment model {DEPLOY}: {SIZE[DEPLOY]:.0f} MB ({SIZE[REF] / SIZE[DEPLOY]:.1f}x smaller than baseline); test delta score vs baseline "
      f"{pp['diff']:+.3f} [{pp['lo']:+.3f}, {pp['hi']:+.3f}]  -> "
      + ("non-inferior within the pre-specified margin" if pp["lo"] > -NONINF_DELTA else "NOT shown non-inferior on test; report honestly"))


Model chosen per outer fold: {0: 'baseline [int8-safe]', 1: 'baseline [int8-safe]', 2: 'baseline [int8-safe]', 3: 'baseline [int8-safe]', 4: 'baseline [int8-safe]'}

DEPLOYMENT CHOICE (all validation data, seed-averaged; margin delta=0.03): baseline [int8-safe]


,name,size_MB,score,diff_vs_ref,lower_bound,noninferior
2,student_flan-t5-small [int8-safe],201.056,0.532,-0.137,-0.186,False
1,student_flan-t5-small [fp32],307.919,0.569,-0.100,-0.137,False
0,baseline [int8-safe],509.263,0.672,0.003,-0.011,True



Procedure-level (per-fold selected model) vs baseline on pooled test:
  score 0.676 [0.649, 0.697] | macro-F1 0.510 [0.460, 0.546] | delta vs baseline +0.001 [-0.010, +0.010]
Deployment model baseline [int8-safe]: 509 MB (1.9x smaller than baseline); test delta score vs baseline +0.001 [-0.010, +0.010]  -> non-inferior within the pre-specified margin


In [17]:
# ---------- ablations (paired bootstrap on pooled test predictions)
rows = []
for name, _, _ in STUDENTS:
    s = "student_" + name.split("/")[-1]; a, b = f"{s} [fp32]", f"{s}_noKD [fp32]"
    if a in names and b in names:
        pp = paired(a, b); rows.append(dict(student=s, params_M=PARAMS[a], score_KD=PT[a]["score"], score_noKD=PT[b]["score"],
                                            delta=pp["diff"], ci_lo=pp["lo"], ci_hi=pp["hi"], p=pp["p"]))
abl_kd = pd.DataFrame(rows)
if len(abl_kd): abl_kd["p_holm"] = holm(abl_kd.p.values)
print("Ablation 1 - does knowledge distillation help? (delta = KD - noKD, composite score)"); display(abl_kd.round(3))

rows = []
for tag in sorted({R[(FOLDS[0], S0)]["cands"][n]["tag"] for n in names if ROLE[n] in ("reference", "compressed")}):
    f = f"{tag} [fp32]"
    if f not in names: continue
    for kind in ("int8", "int8-safe", "int8+emb", "nf4"):
        q = f"{tag} [{kind}]"
        if q in names:
            pp = paired(q, f); rows.append(dict(model=tag, quantizer=kind, size_MB=SIZE[q], score_fp32=PT[f]["score"], score_quant=PT[q]["score"],
                                                delta=pp["diff"], ci_lo=pp["lo"], ci_hi=pp["hi"], format_ok=fmt_ok(q)))
abl_q = pd.DataFrame(rows)
print("\nAblation 2 - quantization effect per model (delta = quantized - fp32)"); display(abl_q.round(3))

rows = []
for n in names:
    if ROLE[n] == "external" or n in ("teacher [fp32]", DEPLOY):
        for ref_name in (REF,):
            pp = paired(n, ref_name); rows.append(dict(model=n, vs=ref_name, delta=pp["diff"], ci_lo=pp["lo"], ci_hi=pp["hi"], p=pp["p"]))
for n in names:
    if ROLE[n] == "external":
        pp = paired(DEPLOY, n); rows.append(dict(model=DEPLOY, vs=n, delta=pp["diff"], ci_lo=pp["lo"], ci_hi=pp["hi"], p=pp["p"]))
abl_ext = pd.DataFrame(rows)
print("\nComparison with teacher / external baselines (paired, composite score)"); display(abl_ext.round(3))

# ---- Ablation 3: does GENERATING explanations change classification accuracy? (same baseline, trained with vs without explanation text)
abl_expl = None
if "baseline_noExpl [fp32]" in names:
    a_, b_ = REF, "baseline_noExpl [fp32]"; pp_ = paired(a_, b_)
    abl_expl = pd.DataFrame([dict(with_explanations=PT[a_]["score"], without_explanations=PT[b_]["score"], delta=pp_["diff"],
                                  ci_lo=pp_["lo"], ci_hi=pp_["hi"], p=pp_["p"])])
    print("\nAblation 3 - classification score with vs without explanation generation (delta = with - without)"); display(abl_expl.round(3))


Ablation 1 - does knowledge distillation help? (delta = KD - noKD, composite score)


,student,params_M,score_KD,score_noKD,delta,ci_lo,ci_hi,p,p_holm
0,student_flan-t5-small,76.961,0.547,0.561,-0.014,-0.075,0.059,0.658,0.658



Ablation 2 - quantization effect per model (delta = quantized - fp32)


,model,quantizer,size_MB,score_fp32,score_quant,delta,ci_lo,ci_hi,format_ok
0,baseline,int8-safe,509.263,0.674,0.676,0.001,-0.010,0.010,1.0
1,student_flan-t5-small,int8-safe,201.056,0.547,0.535,-0.012,-0.051,0.009,1.0



Comparison with teacher / external baselines (paired, composite score)


,model,vs,delta,ci_lo,ci_hi,p
0,baseline [int8-safe],baseline [fp32],0.001,-0.010,0.010,0.856
1,teacher [fp32],baseline [fp32],0.074,0.033,0.110,0.000
2,tfidf-logreg,baseline [fp32],-0.146,-0.185,-0.114,0.000
3,bert-base-uncased,baseline [fp32],0.025,-0.012,0.062,0.188
4,baseline [int8-safe],tfidf-logreg,0.147,0.114,0.184,0.000
5,baseline [int8-safe],bert-base-uncased,-0.024,-0.062,0.018,0.204


In [18]:
# ---------- hyper-parameter search report + seed variance (answers "hyper-parameters were fixed" and "one seed per fold")
sweep = pd.DataFrame([dict(r_, outer_fold=k) for (k, s), r in R.items() if s == S0 for r_ in r.get("lr_sweep", [])])
LR_SWEEP_TAB = LR_CHOSEN_TAB = None
if len(sweep):
    LR_SWEEP_TAB = sweep.groupby(["tag", "lr"]).val_score.agg(["mean", "std", "count"]).reset_index()
    best_rows = sweep.loc[sweep.groupby(["tag", "outer_fold"]).val_score.idxmax()]
    LR_CHOSEN_TAB = best_rows.groupby(["tag", "lr"]).size().rename("folds_choosing_it").reset_index()
    print("Learning-rate search - validation composite score per learning rate (mean/sd over outer folds; each fold used ITS OWN validation projects):")
    display(LR_SWEEP_TAB.round(4))
    print("How often each learning rate was chosen:"); display(LR_CHOSEN_TAB)
    for t, g in LR_SWEEP_TAB.groupby("tag"):
        if len(g) > 1: print(f"  sensitivity of {t}: validation score ranges {g['mean'].min():.3f} .. {g['mean'].max():.3f} across the grid (spread {g['mean'].max() - g['mean'].min():.3f})")
else: print("No learning-rate sweep recorded.")

sd_rows = [dict(model=m, mean_score=PT[m]["score"], seed_sd=seed_sd(m), per_seed=[round(metrics_np(gold_test, ARRS[m]["test"][s][:3])["score"], 3) for s in SEEDS])
           for m in (REF, DEPLOY, "teacher [fp32]", "student_t5-efficient-mini [fp32]") if m in names]
SEED_TAB = pd.DataFrame(sd_rows)
print(f"\nVariation across {len(SEEDS)} training seed(s) (same test set; sd of the pooled composite score):"); display(SEED_TAB.round(4))
if len(SEEDS) > 1:
    typical = float(np.nanmean([r["seed_sd"] for r in sd_rows]))
    print(f"Diagnostic only (do NOT retune delta after seeing results): typical seed sd = {typical:.4f}; pre-specified margin delta = {NONINF_DELTA} "
          f"= {NONINF_DELTA / max(typical, 1e-9):.1f} seed sds.")
else: print("Only one training seed was run: run-to-run variability is NOT quantified. Use TRAIN_SEEDS with >= 3 seeds for publication.")


Learning-rate search - validation composite score per learning rate (mean/sd over outer folds; each fold used ITS OWN validation projects):


,tag,lr,mean,std,count
0,baseline,0.0003,0.6755,0.0650,5
1,student_flan-t5-small,0.0005,0.5566,0.1095,5
2,student_flan-t5-small_noKD,0.0005,0.5885,0.0227,5
3,teacher,0.0003,0.7663,0.0413,5


How often each learning rate was chosen:


,tag,lr,folds_choosing_it
0,baseline,0.0003,5
1,student_flan-t5-small,0.0005,5
2,student_flan-t5-small_noKD,0.0005,5
3,teacher,0.0003,5



Variation across 1 training seed(s) (same test set; sd of the pooled composite score):


,model,mean_score,seed_sd,per_seed
0,baseline [fp32],0.6744,NaN,[0.674]
1,baseline [int8-safe],0.6757,NaN,[0.676]
2,teacher [fp32],0.7482,NaN,[0.748]


Only one training seed was run: run-to-run variability is NOT quantified. Use TRAIN_SEEDS with >= 3 seeds for publication.


In [19]:
# ---------- leakage study: how much would a naive random split inflate the numbers? (cheap TF-IDF + logistic regression, same data)
# This is a contribution in its own right: it quantifies the evaluation bias that project-wise splitting removes.
LEAK_TAB = LEAK_DELTA = None
if RUN_LEAKAGE_STUDY:
    from sklearn.linear_model import LogisticRegression
    cols_l = ["is_functional", "is_quality"] + LABELS

    def cv_tfidf(folds, C=3.0):
        P = np.zeros((len(df), len(cols_l)), int); nd_flag = np.zeros(len(df), bool)
        for f in np.unique(folds):
            te, tr = np.where(folds == f)[0], np.where(folds != f)[0]
            vec_ = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(df.text.iloc[tr])
            Xtr_, Xte_ = vec_.transform(df.text.iloc[tr]), vec_.transform(df.text.iloc[te])
            nd_flag[te] = cosine_similarity(Xte_, Xtr_).max(axis=1) >= NEAR_DUP_THRESH
            for j, c in enumerate(cols_l):
                y = df[c].values[tr]
                if 0 < y.sum() < len(y): P[te, j] = LogisticRegression(C=C, class_weight="balanced", max_iter=2000).fit(Xtr_, y).predict(Xte_)
        return (P[:, 0], P[:, 1], P[:, 2:]), nd_flag

    gold_l = gold_arrays(df); rng_l = np.random.default_rng(SEED + 5)
    schemes = {"project-wise 5-fold (leakage study only)": df.leak_fold.values, "random 5-fold (leaky)": rng_l.permutation(np.arange(len(df)) % LEAK_FOLDS)}
    idx_l = make_boot_indices(len(df), BOOT_N, rng_l, df.project.values)          # SAME resampled projects for both schemes -> paired
    res_l, bt_l, rows_l = {}, {}, []
    for nm, folds_ in schemes.items():
        pred_, nd_flag = cv_tfidf(folds_); res_l[nm] = metrics_np(gold_l, pred_); bt_l[nm] = boot_metrics(gold_l, pred_, idx_l)
        rows_l.append({"split": nm, "score": fmt_ci(res_l[nm]["score"], bt_l[nm]["score"]), "macro-F1": fmt_ci(res_l[nm]["cls_macro_F1"], bt_l[nm]["cls_macro_F1"]),
                       "micro-F1": fmt_ci(res_l[nm]["cls_micro_F1"], bt_l[nm]["cls_micro_F1"]), "test rows with a near-duplicate in train": f"{nd_flag.mean():.1%}"})
    LEAK_TAB = pd.DataFrame(rows_l); display(LEAK_TAB)
    a_, b_ = list(schemes)[1], list(schemes)[0]; dd = bt_l[a_]["score"] - bt_l[b_]["score"]
    LEAK_DELTA = dict(diff=res_l[a_]["score"] - res_l[b_]["score"], lo=ci95(dd)[0], hi=ci95(dd)[1])
    print(f"Inflation from random splitting (score): {res_l[a_]['score'] - res_l[b_]['score']:+.3f} [{ci95(dd)[0]:+.3f}, {ci95(dd)[1]:+.3f}] "
          f"(paired cluster bootstrap). Cheap classifier only; the neural models are expected to show the same direction.")


,split,score,macro-F1,micro-F1,test rows with a near-duplicate in train
0,project-wise 5-fold (leakage study only),"0.583 [0.540, 0.613]","0.418 [0.358, 0.468]","0.502 [0.425, 0.572]",0.8%
1,random 5-fold (leaky),"0.750 [0.715, 0.773]","0.571 [0.514, 0.596]","0.650 [0.591, 0.689]",3.9%


Inflation from random splitting (score): +0.167 [+0.132, +0.204] (paired cluster bootstrap). Cheap classifier only; the neural models are expected to show the same direction.


In [20]:
# ---------- per-class (with CIs), per-project, error analysis  (metrics averaged over training seeds)
def per_class_ci(n):
    tabs = [per_class_table(gold_test, ARRS[n]["test"][s][:3]) for s in SEEDS]
    t = sum(tabs) / len(tabs); t["support"] = tabs[0]["support"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        B = np.array([np.nanmean([per_class_f1(gold_test, ARRS[n]["test"][s][:3], ix)[0] for s in SEEDS], axis=0) for ix in IDX_TEST])
        lo, hi = np.nanpercentile(B, [2.5, 97.5], axis=0)
    t["F1_lo"], t["F1_hi"] = lo, hi; return t
pc_models = [m for m in (REF, DEPLOY, "teacher [fp32]") if m in names]
pc_tab = {m: per_class_ci(m) for m in pc_models}
for m, t in pc_tab.items(): print(f"\nPer-class pooled test results: {m}"); display(t.round(3))

proj_rows = []
for p, idx in test_all.groupby("project").indices.items():
    r = dict(project=p, n=len(idx))
    for m in pc_models: r[m] = seed_metrics(m, "test", gold_test, idx)["score"]
    proj_rows.append(r)
proj_tab = pd.DataFrame(proj_rows).sort_values("project")
print("\nPer-project composite score (generalisation to unseen projects)"); display(proj_tab.round(3))
print("Across projects, mean +- sd:", {m: f"{proj_tab[m].mean():.3f} +- {proj_tab[m].std():.3f}" for m in pc_models})

# error analysis for the deployment model (first training seed)
fr_, q_, P_ = ARR[DEPLOY]["test"][:3]; gY = gold_test[2]
err_rows = []
for i in range(len(test_all)):
    miss = [LABELS[j] for j in range(len(LABELS)) if gY[i, j] and not P_[i, j]]; extra = [LABELS[j] for j in range(len(LABELS)) if P_[i, j] and not gY[i, j]]
    if miss or extra or fr_[i] != gold_test[0][i] or q_[i] != gold_test[1][i]:
        err_rows.append(dict(project=test_all.project[i], text=test_all.text[i], missed=miss, spurious=extra,
                             FR_err=int(fr_[i] != gold_test[0][i]), NFR_err=int(q_[i] != gold_test[1][i])))
err_df = pd.DataFrame(err_rows)
pairs = Counter((m, e) for r in err_rows for m in r["missed"] for e in r["spurious"])
print(f"\nError analysis ({DEPLOY}, seed {S0}): {len(err_df)}/{len(test_all)} requirements with any error")
print("Most frequent confusions (gold class missed -> class predicted instead):", pairs.most_common(8))
display(err_df.sample(min(10, len(err_df)), random_state=0) if len(err_df) else err_df)



Per-class pooled test results: baseline [fp32]


,P,R,F1,support,F1_lo,F1_hi
Availability,0.667,0.457,0.542,35,0.381,0.711
Fault Tolerance,0.364,0.250,0.296,16,0.000,0.526
Legal,0.621,0.581,0.600,31,0.279,0.739
Look & Feel,0.582,0.582,0.582,67,0.465,0.698
Maintainability,0.293,0.353,0.320,34,0.213,0.471
Operability,0.510,0.448,0.477,58,0.309,0.597
Performance,0.733,0.524,0.611,63,0.475,0.705
Portability,0.400,0.455,0.426,22,0.210,0.561
Scalability,0.654,0.586,0.618,29,0.449,0.731
Security,0.768,0.573,0.656,75,0.589,0.773



Per-class pooled test results: baseline [int8-safe]


,P,R,F1,support,F1_lo,F1_hi
Availability,0.619,0.371,0.464,35,0.329,0.636
Fault Tolerance,0.333,0.188,0.240,16,0.000,0.480
Legal,0.487,0.613,0.543,31,0.210,0.706
Look & Feel,0.550,0.493,0.520,67,0.368,0.643
Maintainability,0.275,0.412,0.329,34,0.190,0.517
Operability,0.500,0.517,0.508,58,0.378,0.612
Performance,0.805,0.524,0.635,63,0.488,0.731
Portability,0.455,0.455,0.455,22,0.240,0.571
Scalability,0.643,0.621,0.632,29,0.459,0.767
Security,0.768,0.707,0.736,75,0.647,0.836



Per-class pooled test results: teacher [fp32]


,P,R,F1,support,F1_lo,F1_hi
Availability,0.722,0.743,0.732,35,0.607,0.831
Fault Tolerance,0.286,0.375,0.324,16,0.105,0.526
Legal,0.613,0.613,0.613,31,0.387,0.727
Look & Feel,0.614,0.642,0.628,67,0.488,0.752
Maintainability,0.417,0.441,0.429,34,0.273,0.577
Operability,0.717,0.569,0.635,58,0.500,0.745
Performance,0.804,0.587,0.679,63,0.545,0.766
Portability,0.550,0.500,0.524,22,0.000,0.667
Scalability,0.900,0.621,0.735,29,0.526,0.895
Security,0.747,0.747,0.747,75,0.606,0.889



Per-project composite score (generalisation to unseen projects)


,project,n,baseline [fp32],baseline [int8-safe],teacher [fp32]
0,1,28,0.731,0.699,0.664
1,10,52,0.652,0.609,0.735
2,11,13,0.604,0.614,0.729
3,12,22,0.634,0.648,0.670
4,13,19,0.519,0.505,0.540
5,14,16,0.540,0.537,0.566
6,15,12,0.530,0.559,0.597
7,2,38,0.696,0.697,0.752
8,3,79,0.591,0.608,0.789
9,4,55,0.648,0.632,0.666


Across projects, mean +- sd: {'baseline [fp32]': '0.637 +- 0.075', 'baseline [int8-safe]': '0.640 +- 0.075', 'teacher [fp32]': '0.708 +- 0.089'}

Error analysis (baseline [int8-safe], seed 42): 308/617 requirements with any error
Most frequent confusions (gold class missed -> class predicted instead): [(('Portability', 'Operability'), 6), (('Operability', 'Maintainability'), 5), (('Availability', 'Maintainability'), 5), (('Maintainability', 'Usability'), 5), (('Fault Tolerance', 'Maintainability'), 4), (('Operability', 'Portability'), 4), (('Operability', 'Scalability'), 4), (('Availability', 'Scalability'), 4)]


,project,text,missed,spurious,FR_err,NFR_err
63,3,2 out of 3 Program Administrators/Nursing Staf...,[],[],1,0
247,4,100% of the cardmember and merchant services r...,[Scalability],[],0,0
293,10,The product shall continue to assign turns unt...,[],"[Maintainability, Usability]",1,1
124,6,The product must work with most database manag...,[Portability],[],0,0
188,2,The product shall generate a CMA report in an ...,[Performance],[],0,1
144,6,A new user must be able to be added to the sys...,[],[Usability],0,0
299,14,A new user shall be able to navigate through t...,"[Look & Feel, Performance, Usability]",[],1,1
65,3,The verbiage used within the system shall be c...,[Look & Feel],[Legal],0,0
26,8,All aspects of the website shall be accessed b...,[Operability],[Portability],0,0
66,3,The system shall be available for use between ...,[Availability],[Maintainability],0,0


## 7. Explanation quality

Four complementary signals (none replaces human judgement): **ROUGE-L** and **BERTScore** against the Groq-written reference (measures imitation of the reference, not truth); **label consistency** (the explanation names exactly the categories the model predicted — a cheap faithfulness check); **LLM-judge** scores on a random sample; and a **blinded human-evaluation sheet** that must be filled in by at least two annotators (report inter-annotator agreement).

**v3:** the explanations are post-hoc rationales imitating an LLM that saw the gold labels, not evidence of reasoning. Added: a reference-free evidence-grounding check, a blinded human sheet (3 systems incl. a gold control) with an analysis cell (Cohen's kappa), and an optional judge that is a different Groq model than the reference writer.

In [21]:
from rouge_score import rouge_scorer
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
EXPL_MODELS = [m for m in (REF, DEPLOY, "teacher [fp32]", "student_t5-efficient-mini [fp32]") if m in names]
expl_tab = None
if not EXPL_VALID:
    print("Explanation metrics DISABLED: targets were templates.")
else:
    scorer_b = None
    try:
        from bert_score import BERTScorer
        scorer_b = BERTScorer(lang="en", model_type="roberta-large", rescale_with_baseline=True, device=DEVICE, batch_size=32)
    except Exception as e: print("BERTScore skipped:", type(e).__name__, str(e)[:150])
    VEC = {}
    for m in EXPL_MODELS:
        per_seed = []
        for s in SEEDS:
            ok, expl, Y = ARRS[m]["test"][s][3], ARRS[m]["test"][s][4], ARRS[m]["test"][s][2]
            d_ = dict(rl=np.array([_rouge.score(ref, e)["rougeL"].fmeasure for ref, e in zip(test_all.explanation, expl)]),
                      cons=np.array([label_consistency(e, [l for l, v in zip(LABELS, Y[i]) if v]) for i, e in enumerate(expl)], float),
                      ok=ok.astype(float))
            ev = np.array([evidence_counts(e, r) for e, r in zip(expl, test_all.text)], float).reshape(-1, 2)
            d_["ev_n"], d_["ev_g"] = ev[:, 0], ev[:, 1]
            if scorer_b is not None:
                _, _, F1b = scorer_b.score([e if e.strip() else "none" for e in expl], test_all.explanation.tolist()); d_["bs"] = F1b.numpy()
            per_seed.append(d_)
        VEC[m] = {k: np.mean([d_[k] for d_ in per_seed], axis=0) for k in per_seed[0]}
    rows = []
    for m in EXPL_MODELS:
        v = VEC[m]; r = dict(model=m)
        mean_ci = lambda arr: fmt_ci(arr.mean(), np.array([arr[ix].mean() for ix in IDX_TEST]))
        r["ROUGE-L"] = mean_ci(v["rl"]); r["label-consistency"] = mean_ci(v["cons"]); r["format_ok"] = mean_ci(v["ok"])
        if "bs" in v: r["BERTScore-F1"] = mean_ci(v["bs"])
        # evidence grounding: of the phrases the explanation puts in quotation marks, how many really occur in the requirement (micro average)
        eg = np.array([v["ev_g"][ix].sum() / max(v["ev_n"][ix].sum(), 1e-9) for ix in IDX_TEST])
        r["quoted evidence found in requirement"] = fmt_ci(v["ev_g"].sum() / max(v["ev_n"].sum(), 1e-9), eg)
        r["quotes per explanation"] = round(float(v["ev_n"].mean()), 2)
        rows.append(r)
    expl_tab = pd.DataFrame(rows); display(expl_tab)
    print("Reference = LLM-written explanation (Groq) written FROM the gold labels, so these are post-hoc rationales, not evidence of model reasoning. "
          "ROUGE/BERTScore = imitation of that reference; label-consistency = agreement with the model's OWN predicted categories; "
          "evidence = cheap, reference-free check that quoted phrases are real. Averaged over training seeds. Human ratings (next cells) are the primary evidence.")


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,model,ROUGE-L,label-consistency,format_ok,BERTScore-F1,quoted evidence found in requirement,quotes per explanation
0,baseline [fp32],"0.414 [0.379, 0.450]","0.872 [0.845, 0.898]","1.000 [1.000, 1.000]","0.445 [0.415, 0.472]","0.858 [0.798, 0.896]",0.81
1,baseline [int8-safe],"0.423 [0.397, 0.450]","0.853 [0.826, 0.878]","1.000 [1.000, 1.000]","0.453 [0.431, 0.475]","0.865 [0.819, 0.903]",0.88
2,teacher [fp32],"0.416 [0.383, 0.453]","0.859 [0.815, 0.900]","1.000 [1.000, 1.000]","0.457 [0.425, 0.486]","0.880 [0.847, 0.918]",0.78


Reference = LLM-written explanation (Groq) written FROM the gold labels, so these are post-hoc rationales, not evidence of model reasoning. ROUGE/BERTScore = imitation of that reference; label-consistency = agreement with the model's OWN predicted categories; evidence = cheap, reference-free check that quoted phrases are real. Averaged over training seeds. Human ratings (next cells) are the primary evidence.


In [22]:
judge_tab = None; HUMAN_SHEET = HUMAN_KEY = None; JUDGE_SCORES = {}; HUMAN_PICK = []
SYS3 = [REF, DEPLOY, "GOLD-REFERENCE"]        # GOLD-REFERENCE = gold labels + Groq-written explanation: a control that annotators should rate highest
HUMAN_INSTR = """Human evaluation instructions
For every row you see a requirement, reference labels and THREE anonymous outputs (output_1..3). Fill in:
  lab_1..lab_3   (1-5): do the predicted functional/quality flags and NFR categories agree with the reference labels? 5 = fully agree, 1 = completely wrong
  expl_1..expl_3 (1-5): is the explanation correct, grounded in words that really occur in the requirement, and clear? 5 = excellent, 1 = wrong or invented
  best_output    : 1, 2, 3, or none  (which output is best overall; do not guess who produced what)
Work independently. Save your copy as annotator1.csv / annotator2.csv (never edit the 'id' column) in the human_eval folder."""
if EXPL_VALID:
    rs = np.random.default_rng(SEED + 7); n_h = min(100, len(test_all)); pick = rs.choice(len(test_all), n_h, replace=False); HUMAN_PICK = [int(i) for i in pick]
    def sys_output(sysname, i): return make_target(test_all.iloc[i]) if sysname == "GOLD-REFERENCE" else ARR[sysname]["test"][5][i]
    sheet, key = [], []
    for j, i in enumerate(pick):
        order = rs.permutation(3); outs = [SYS3[o] for o in order]
        sheet.append(dict(id=j, requirement=test_all.text[i], reference_labels=make_target(test_all.iloc[i]).split("| explanation:")[0].strip(),
                          output_1=sys_output(outs[0], i), output_2=sys_output(outs[1], i), output_3=sys_output(outs[2], i),
                          lab_1="", lab_2="", lab_3="", expl_1="", expl_2="", expl_3="", best_output=""))
        key.append(dict(id=j, sys_1=outs[0], sys_2=outs[1], sys_3=outs[2]))
    HUMAN_SHEET, HUMAN_KEY = pd.DataFrame(sheet), pd.DataFrame(key)

    # ---------- LLM judge: a DIFFERENT model than the one that wrote the references, scoring each system independently and blind, on the same items
    if RUN_LLM_JUDGE and oai is not None and JUDGE_MODEL_USED:
        judge_model = JUDGE_MODEL_USED                                      # a Groq model (chosen in section 3)
        JUDGE_PROMPT = """You are an expert requirements engineer grading an automatic classifier's output. The system is anonymous.
Requirement: "{text}"
Reference labels: {gold}
System output: {out}
Rate each from 1 (poor) to 5 (excellent):
correctness - do the predicted type and NFR categories agree with the reference labels?
faithfulness - does the explanation cite words that really appear in the requirement and justify the predicted labels, without inventing facts?
clarity - is the explanation clear and concise?
Return ONLY JSON: {{"correctness": int, "faithfulness": int, "clarity": int}}"""
        def judge_one(text, gold, out):
            msg = JUDGE_PROMPT.format(text=text, gold=gold, out=out)
            for a in range(3):
                try:
                    txt, _ = _chat(msg, max_out=400, est_tokens=len(msg) / 3.5 + 250, model=judge_model)
                    d = json.loads(re.search(r"\{.*?\}", txt, re.S).group(0)); return {k: int(d[k]) for k in ("correctness", "faithfulness", "clarity")}
                except DailyQuota: return None
                except Exception: time.sleep(2 ** a)
            return None
        sample = pick[:min(JUDGE_N, len(pick))]                                   # same items as the human sheet -> judge/human agreement can be computed
        jm = SYS3 + (["teacher [fp32]"] if "teacher [fp32]" in names else [])
        for m in jm:
            JUDGE_SCORES[m] = {}
            for i in tqdm(sample, desc=f"judge {m}", leave=False):
                gold = make_target(test_all.iloc[i]).split("| explanation:")[0].strip()
                r_ = judge_one(test_all.text[i], gold, sys_output(m, i))
                if r_ is not None: JUDGE_SCORES[m][int(i)] = r_
        common = sorted(set.intersection(*[set(JUDGE_SCORES[m]) for m in jm]))
        if common:
            ci_ = np.array(common); bi = np.random.default_rng(SEED + 12).integers(0, len(ci_), (BOOT_N, len(ci_))); rows = []
            for m in jm:
                r = dict(model=m, n=len(ci_))
                for k_ in ("correctness", "faithfulness", "clarity"):
                    v = np.array([JUDGE_SCORES[m][i][k_] for i in ci_], float); r[k_] = fmt_ci(v.mean(), v[bi].mean(1))
                rows.append(r)
            judge_tab = pd.DataFrame(rows); display(judge_tab)
            print(f"Judge model: {judge_model}; writer of references: {WRITER_MODEL}. GOLD-REFERENCE is a sanity control (should score highest). "
                  "Supporting evidence only: compare with the human ratings in the next cell.")
    elif RUN_LLM_JUDGE: print("LLM judge skipped: no Groq client / judge model available.")
    print(f"Human sheet prepared: {len(HUMAN_SHEET)} items x 3 anonymous outputs (baseline, deployment model, gold control).")


Human sheet prepared: 100 items x 3 anonymous outputs (baseline, deployment model, gold control).


In [23]:
# ---------- HUMAN EVALUATION: write the blank sheet, then analyse the annotators' filled copies when they exist
# Workflow: (1) run this cell once -> nice_work_v3/human_eval/sheet_blank.csv  (2) give a copy to each annotator, who save it as
# annotator1.csv, annotator2.csv (...) in that folder  (3) re-run THIS cell only: it computes ratings, CIs, win rates and inter-annotator kappa.
HUMAN_RESULTS, HUMAN_TAB = None, None
os.makedirs(HUMAN_EVAL_DIR, exist_ok=True)
if not EXPL_VALID or HUMAN_SHEET is None:
    print("No valid explanation targets -> human evaluation not applicable.")
else:
    HUMAN_SHEET.to_csv(os.path.join(HUMAN_EVAL_DIR, "sheet_blank.csv"), index=False)
    HUMAN_KEY.to_csv(os.path.join(HUMAN_EVAL_DIR, "KEY_do_not_share.csv"), index=False)
    open(os.path.join(HUMAN_EVAL_DIR, "INSTRUCTIONS.txt"), "w").write(HUMAN_INSTR)
    files = sorted(f for f in os.listdir(HUMAN_EVAL_DIR) if re.fullmatch(r"annotator\d+\.csv", f))
    if len(files) < 2:
        print(f"Human evaluation NOT done yet: {len(files)} completed annotator file(s) found in {HUMAN_EVAL_DIR}. Give sheet_blank.csv to >= 2 people, "
              "save their copies as annotator1.csv, annotator2.csv, then re-run this cell. Do not claim explanation quality before that.")
    else:
        from sklearn.metrics import cohen_kappa_score
        from scipy.stats import spearmanr
        key = HUMAN_KEY.set_index("id"); long = []
        for f in files:
            a = pd.read_csv(os.path.join(HUMAN_EVAL_DIR, f)).set_index("id")
            for i in a.index:
                best = str(a.loc[i, "best_output"]).strip().replace(".0", "")
                for p in (1, 2, 3):
                    long.append(dict(annotator=f[:-4], id=int(i), pos=p, system=key.loc[int(i), f"sys_{p}"],
                                     lab=pd.to_numeric(a.loc[i, f"lab_{p}"], errors="coerce"), expl=pd.to_numeric(a.loc[i, f"expl_{p}"], errors="coerce"),
                                     best=int(best == str(p))))
        L = pd.DataFrame(long); miss = float(L[["lab", "expl"]].isna().any(axis=1).mean())
        assert miss < 0.05, f"{miss:.0%} of the ratings are empty/non-numeric; annotators must complete the sheet"
        L = L.dropna(subset=["lab", "expl"]); L[["lab", "expl"]] = L[["lab", "expl"]].astype(int)
        assert L[["lab", "expl"]].stack().between(1, 5).all(), "ratings must be integers 1-5"
        ann_names = sorted(L.annotator.unique()); kap = {}
        for metric, col in (("labels", "lab"), ("explanation", "expl")):
            ks = []
            for x in range(len(ann_names)):
                for y in range(x + 1, len(ann_names)):
                    A = L[L.annotator == ann_names[x]].set_index(["id", "pos"])[col]; B = L[L.annotator == ann_names[y]].set_index(["id", "pos"])[col]
                    j = pd.concat([A, B], axis=1, keys=["a", "b"]).dropna()
                    ks.append(cohen_kappa_score(j.a, j.b, weights="quadratic"))
            kap[f"weighted kappa ({metric})"] = float(np.mean(ks))
        bs_ = L[L.best == 1].groupby(["annotator", "id"]).system.first().unstack("annotator")
        ks = []
        for x in range(len(ann_names)):
            for y in range(x + 1, len(ann_names)):
                j = bs_[[ann_names[x], ann_names[y]]].dropna()
                if len(j) > 5: ks.append(cohen_kappa_score(j.iloc[:, 0], j.iloc[:, 1]))
        kap["kappa (best output)"] = float(np.mean(ks)) if ks else float("nan")
        print("Inter-annotator agreement:", {k: round(v, 3) for k, v in kap.items()}, "| annotators:", ann_names)
        if min(v for v in kap.values() if not np.isnan(v)) < 0.4: print("WARNING: agreement below 0.40 -> refine the guidelines / add an adjudication round before reporting.")

        M = L.groupby(["id", "system"])[["lab", "expl"]].mean().unstack("system").dropna()
        n_it = len(M); bi = np.random.default_rng(SEED + 21).integers(0, n_it, (BOOT_N, n_it)); rows = []
        for sy in SYS3:
            r = dict(system=sy, items=n_it)
            for col, lab in (("lab", "label agreement (1-5)"), ("expl", "explanation quality (1-5)")):
                v = M[col][sy].values; r[lab] = fmt_ci(v.mean(), v[bi].mean(1))
            r["chosen best (share)"] = round(float((L[(L.best == 1)].system == sy).sum() / max(L.groupby("annotator").id.nunique().sum(), 1)), 3)
            rows.append(r)
        HUMAN_TAB = pd.DataFrame(rows); display(HUMAN_TAB)
        d_ = M["expl"][DEPLOY].values - M["expl"][REF].values; boots = d_[bi].mean(1)
        HUMAN_RESULTS = dict(kappa=kap, deploy_minus_baseline_expl=float(d_.mean()), ci=ci95(boots), p=boot_pvalue(boots), n_items=n_it)
        print(f"Explanation quality, deployment minus baseline: {d_.mean():+.2f} [{ci95(boots)[0]:+.2f}, {ci95(boots)[1]:+.2f}], p={boot_pvalue(boots):.3f}")
        if M["expl"]["GOLD-REFERENCE"].mean() < max(M["expl"][REF].mean(), M["expl"][DEPLOY].mean()): print("WARNING: the gold control was NOT rated highest -> check annotator reliability.")
        if JUDGE_SCORES:
            xs, ys = [], []
            for sy in SYS3:
                for j in M.index:
                    i = HUMAN_PICK[int(j)]
                    if sy in JUDGE_SCORES and i in JUDGE_SCORES[sy]: xs.append(M["expl"][sy].loc[j]); ys.append(JUDGE_SCORES[sy][i]["faithfulness"])
            if len(xs) > 10: print(f"LLM-judge vs human (Spearman rho on explanation quality): {spearmanr(xs, ys)[0]:.2f}  (n={len(xs)})")


Human evaluation NOT done yet: 0 completed annotator file(s) found in nice_conference_efficient/human_eval. Give sheet_blank.csv to >= 2 people, save their copies as annotator1.csv, annotator2.csv, then re-run this cell. Do not claim explanation quality before that.


In [24]:
# ---------- latency: batch size 1, warm-up, fixed thread count; CPU for fp32/int8, GPU for nf4 (labelled)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
LAT_FOLD = FOLDS[0]; lat_srcs = R[(LAT_FOLD, S0)]["test_df"].src.tolist()[:LAT_N]
def time_model(m, device):
    for s in lat_srcs[:3]: generate_seq2seq(m, tok, [s], bs=1)
    ts = []
    for s in lat_srcs:
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter(); generate_seq2seq(m, tok, [s], bs=1)
        if device == "cuda": torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts)), float(np.percentile(ts, 90))
rows = []
for n in names:
    c = R[(LAT_FOLD, S0)]["cands"][n]
    if c.get("path") is None or not os.path.isdir(c["path"]) or c["role"] in ("teacher", "ablation"): continue
    m = None
    try:
        if c["kind"] == "nf4": m = load_nf4(c["path"]); med, p90 = time_model(m, "cuda"); dev = "gpu"
        else:
            m = (_fp32(c["path"]).eval() if c["kind"] == "fp32" else LOADERS[c["kind"]](c["path"])); med, p90 = time_model(m, "cpu"); dev = f"cpu x{torch.get_num_threads()}"
        rows.append(dict(model=n, device=dev, median_ms=med, p90_ms=p90))
        if c["kind"] == "fp32" and torch.cuda.is_available():
            m = m.to("cuda"); med, p90 = time_model(m, "cuda"); rows.append(dict(model=n, device="gpu", median_ms=med, p90_ms=p90))
    except Exception as e: print(f"latency skipped for {n}: {type(e).__name__}: {str(e)[:120]}")
    finally: del m; free()
lat_tab = pd.DataFrame(rows); display(lat_tab.round(1))
print("Teacher weights are deleted after distillation, so teacher latency is not measured (set KEEP_MODELS / keep it if you need it).")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


,model,device,median_ms,p90_ms
0,baseline [fp32],cpu x4,2945.0,3739.5
1,baseline [fp32],gpu,1288.1,1676.2
2,baseline [int8-safe],cpu x4,2882.3,3787.2
3,student_flan-t5-small [fp32],cpu x4,1527.5,2056.0
4,student_flan-t5-small [fp32],gpu,963.8,1426.6
5,student_flan-t5-small [int8-safe],cpu x4,2165.6,3026.1


Teacher weights are deleted after distillation, so teacher latency is not measured (set KEEP_MODELS / keep it if you need it).


In [25]:
# ---------- EXTERNAL VALIDATION on a SECOND dataset (optional; set EXTERNAL_DATASET["path"] in the config cell)
# Every fold/seed model (trained on PROMISE only) is applied to the second dataset. Only classes present in BOTH label sets are scored
# (predictions for other classes are ignored); functional/quality flags are always scored. Near-duplicates of PROMISE text are removed.
EXT = None
def load_external(cfg):
    d = None
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try: d = pd.read_csv(cfg["path"], encoding=enc); break
        except UnicodeDecodeError: continue
    out = pd.DataFrame({"text": d[cfg["text_col"]].astype(str).str.strip().str.strip("'\"").str.strip()})
    out["project"] = d[cfg["group_col"]].astype(str).values if cfg.get("group_col") else "ext-all"
    out["is_functional"] = d[cfg["fr_col"]].astype(int).values; out["is_quality"] = d[cfg["q_col"]].astype(int).values
    for l in LABELS: out[l] = 0
    for col, lab in cfg.get("class_cols", {}).items():
        assert lab in LABELS, f"unknown label {lab}"; out[lab] = np.maximum(out[lab].values, d[col].fillna(0).astype(int).values)
    if cfg.get("class_col"):
        for i, v in enumerate(d[cfg["class_col"]].fillna("").astype(str)):
            for part in v.split(cfg["class_sep"]):
                lab = cfg["class_value_map"].get(part.strip())
                if lab in LABELS: out.loc[i, lab] = 1
    mapped = sorted((set(cfg.get("class_cols", {}).values()) | set(cfg.get("class_value_map", {}).values())) & set(LABELS))
    out["_key"] = out.text.str.lower().str.replace(r"[^a-z0-9 ]", " ", regex=True).str.split().str.join(" ")
    out = out[out._key.str.len() > 0].drop_duplicates(subset="_key").reset_index(drop=True)
    return out, mapped

if EXTERNAL_DATASET.get("path") and os.path.exists(EXTERNAL_DATASET["path"]):
    ext_df, ext_labels = load_external(EXTERNAL_DATASET)
    n_raw = len(ext_df)
    vec_e = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(df.text)
    sim = cosine_similarity(vec_e.transform(ext_df.text), vec_e.transform(df.text)).max(axis=1)
    ext_df = ext_df[sim < NEAR_DUP_THRESH].reset_index(drop=True); ext_df["src"] = ext_df.text.map(make_input)
    print(f"External dataset '{EXTERNAL_DATASET['name']}': {n_raw} requirements, {n_raw - len(ext_df)} removed as near-duplicates of PROMISE -> {len(ext_df)} | "
          f"scored classes: {ext_labels} | groups: {ext_df.project.nunique()}")
    gold_e = gold_arrays(ext_df); mask_cols = np.array([int(l in ext_labels) for l in LABELS])
    EXT_PRED = {}; ext_models = [REF, DEPLOY]; insts = [(k, s) for s in SEEDS for k in FOLDS]
    for n in ext_models:
        for (k, s) in insts:
            c = R[(k, s)]["cands"][n]; m = None
            try:
                m = LOADERS[c["kind"]](c["path"])
                raw = generate_seq2seq(m, tok, ext_df.src.tolist(), bs=32 if str(m.device).startswith("cuda") else 8)
                fr_e, q_e, P_e = parsed_to_arrays([parse_output(r) for r in raw]); EXT_PRED[(n, k, s)] = (fr_e, q_e, P_e * mask_cols)
            finally: del m; free()
    groups_e = ext_df.project.values if ext_df.project.nunique() >= 5 else None          # cluster bootstrap only if there are enough groups
    idx_e = make_boot_indices(len(ext_df), BOOT_N, np.random.default_rng(SEED + 3), groups_e)
    def ext_point(n): return {k_: float(np.mean([metrics_np(gold_e, EXT_PRED[(n, k, s)])[k_] for (k, s) in insts])) for k_ in KEYS}
    def ext_boot(n):
        bs_ = [boot_metrics(gold_e, EXT_PRED[(n, k, s)], idx_e, KEYS) for (k, s) in insts]
        return {k_: np.mean([b[k_] for b in bs_], axis=0) for k_ in KEYS}
    PT_E = {n: ext_point(n) for n in ext_models}; BT_E = {n: ext_boot(n) for n in ext_models}
    rows = []
    for n in ext_models:
        sc_ = [metrics_np(gold_e, EXT_PRED[(n, k, s)])["score"] for (k, s) in insts]
        rows.append(dict(model=n, score=fmt_ci(PT_E[n]["score"], BT_E[n]["score"]), FR_F1=fmt_ci(PT_E[n]["ident_FR_F1"], BT_E[n]["ident_FR_F1"]),
                         NFR_F1=fmt_ci(PT_E[n]["ident_NFR_F1"], BT_E[n]["ident_NFR_F1"]), macro_F1=fmt_ci(PT_E[n]["cls_macro_F1"], BT_E[n]["cls_macro_F1"]),
                         sd_across_models=round(float(np.std(sc_, ddof=1)) if len(sc_) > 1 else float("nan"), 4)))
    dE = BT_E[DEPLOY]["score"] - BT_E[REF]["score"]; lo_e, hi_e = ci95(dE)
    EXT = dict(table=pd.DataFrame(rows), delta=PT_E[DEPLOY]["score"] - PT_E[REF]["score"], lo=lo_e, hi=hi_e, noninferior=bool(float(np.percentile(dE, 5)) > -NONINF_DELTA),
               n=len(ext_df), labels=ext_labels)
    display(EXT["table"])
    print(f"External-dataset delta (deployment - baseline): {EXT['delta']:+.3f} [{lo_e:+.3f}, {hi_e:+.3f}] -> "
          + ("non-inferior within the pre-specified margin" if EXT["noninferior"] else "NOT shown non-inferior on the external data"))
else:
    print("External dataset NOT configured -> skipped. For a stronger venue, set EXTERNAL_DATASET (path, text_col, fr_col, q_col, class_cols / class_col) in the config cell "
          "and re-run from this cell. Without it, claims are limited to PROMISE.")


External dataset NOT configured -> skipped. For a stronger venue, set EXTERNAL_DATASET (path, text_col, fr_col, q_col, class_cols / class_col) in the config cell and re-run from this cell. Without it, claims are limited to PROMISE.


In [26]:
# ---------- REFIT the deployment model on ALL projects (the released model should not be one fold's 4/5 model)
# No held-out data remains, so: epochs = median best epoch of the CV runs, learning rate = most frequently chosen one (both from CV validation only),
# same oversampling and the same distillation recipe (teacher refit on all projects too, if the deployment model is a KD student).
REFIT_PATH, REFIT_INFO = None, {}
if REFIT_FINAL:
    refit_dir = os.path.join(WORK_DIR, "refit"); final_dir = os.path.join(refit_dir, "final_fp32")
    def _vals(key, t):
        v = [r[key][t] for r in R.values() if t in r.get(key, {})]
        assert v, f"no cross-validation record of '{key}' for '{t}'; cannot refit"
        return v
    mode_lr = lambda t: Counter(_vals("lr_used", t)).most_common(1)[0][0]
    med_ep = lambda t: max(2, int(round(float(np.median(_vals("best_epoch", t))))))
    if DEPLOY_TAG == "baseline": d_name, d_bs = BASELINE_NAME, 8
    else: d_name, d_bs = next(n for n, _, _ in STUDENTS if "student_" + n.split("/")[-1] == DEPLOY_TAG), 16
    REFIT_INFO = dict(tag=DEPLOY_TAG, kind=DEPLOY_KIND, epochs=med_ep(DEPLOY_TAG), lr=mode_lr(DEPLOY_TAG), projects=int(df.project.nunique()), n_requirements=len(df))
    if os.path.isdir(final_dir): print("Refit model already exists -> reusing", final_dir)
    else:
        full = oversample_rare(df, LABELS, MIN_CLASS_COUNT, SEED) if OVERSAMPLE_RARE else df
        full_ds = tokenize_df(full); teacher_obj = None
        if DEPLOY_TAG.startswith("student_"):
            used = Counter(r["leak"]["teacher_used"] for r in R.values()).most_common(1)[0][0]
            if used == "teacher": t_name, t_tag, t_bs, t_acc, t_ck = TEACHER_NAME, "teacher", 4, 2, True
            else: t_name, t_tag, t_bs, t_acc, t_ck = BASELINE_NAME, "baseline", 8, 1, False
            t_path = train_fixed(t_name, "teacher_full", full_ds, refit_dir, med_ep(t_tag), mode_lr(t_tag), bs=t_bs, grad_acc=t_acc, grad_ckpt=t_ck, seed=SEED)
            teacher_obj = _fp32(t_path).to(DEVICE).eval(); REFIT_INFO["teacher"] = t_tag
        final_dir = train_fixed(d_name, "final_fp32", full_ds, refit_dir, REFIT_INFO["epochs"], REFIT_INFO["lr"], bs=d_bs, teacher=teacher_obj, seed=SEED)
        del teacher_obj; free()
        if os.path.isdir(os.path.join(refit_dir, "teacher_full")): shutil.rmtree(os.path.join(refit_dir, "teacher_full"), ignore_errors=True)
    REFIT_PATH = final_dir
    # smoke test (NOT an accuracy estimate: the model has seen these rows): output format on 100 random requirements
    sub = df.sample(min(100, len(df)), random_state=0); m_ = LOADERS[DEPLOY_KIND](REFIT_PATH)
    REFIT_INFO["format_ok_smoke"] = score_raw(generate_seq2seq(m_, tok, sub.src.tolist(), bs=16), sub)["format_ok"]; del m_; free()
    print("Refit:", REFIT_INFO)
    if EXT is not None:      # the only genuinely out-of-sample check of the refit model
        m_ = LOADERS[DEPLOY_KIND](REFIT_PATH)
        fr_r, q_r, P_r = parsed_to_arrays([parse_output(r) for r in generate_seq2seq(m_, tok, ext_df.src.tolist(), bs=16)]); del m_; free()
        sc_r = metrics_np(gold_e, (fr_r, q_r, P_r * mask_cols))["score"]; REFIT_INFO["external_score"] = sc_r
        print(f"Refit model on the external dataset: score {sc_r:.3f} (fold models: {PT_E[DEPLOY]['score']:.3f} mean).")
    else: print("No external dataset: the refit model has NO independent evaluation. Its expected quality is the CV estimate of the same recipe; say so in the paper.")
else: print("REFIT_FINAL=False -> the released model will be the best fold's model (trained on ~4/5 of the projects).")


REFIT_FINAL=False -> the released model will be the best fold's model (trained on ~4/5 of the projects).


In [27]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for n in names:
    if ROLE[n] not in ("reference", "compressed", "teacher", "external"): continue
    lo, hi = ci95(BT[n]["score"]); col = {"reference": "k", "teacher": "tab:purple", "external": "tab:gray"}.get(ROLE[n], "tab:blue")
    ax.errorbar(SIZE[n], PT[n]["score"], yerr=[[PT[n]["score"] - lo], [hi - PT[n]["score"]]], fmt="o", color=("tab:red" if n == DEPLOY else col),
                ms=8 if n == DEPLOY else 5, capsize=2, alpha=0.9)
    if ROLE[n] != "compressed" or n == DEPLOY or n.endswith("[fp32]"): ax.annotate(n.replace("student_", ""), (SIZE[n], PT[n]["score"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axhline(PT[REF]["score"] - NONINF_DELTA, ls="--", c="gray", lw=0.8); ax.text(ax.get_xlim()[0], PT[REF]["score"] - NONINF_DELTA, " non-inferiority margin", fontsize=7, va="bottom")
ax.set_xscale("log"); ax.set_xlabel("serialized size (MB, log)"); ax.set_ylabel("composite score (pooled out-of-fold test, 95% CI)")
ax.set_title("Quality vs size (red = deployment choice)"); plt.tight_layout()
fig.savefig(os.path.join(WORK_DIR, "pareto.png"), dpi=200); plt.show()

/tmp/ipykernel_50/1613856635.py:10: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  ax.set_title("Quality vs size (red = deployment choice)"); plt.tight_layout()


In [28]:
# ---------- claims guard: the only claims these results support (generated from the numbers, decided by the PRE-SPECIFIED rules)
pp = paired(DEPLOY, REF)
p_h = float(headline.loc[headline.model == DEPLOY, "p_holm_vs_base"].iloc[0]) if (headline.model == DEPLOY).any() and "p_holm_vs_base" in headline else float("nan")
CLAIMS, NOT_CLAIMS = [], []
CLAIMS.append(f"Compression: {DEPLOY} is {SIZE[REF] / SIZE[DEPLOY]:.1f}x smaller than the {BASELINE_NAME} baseline "
              f"({SIZE[DEPLOY]:.0f} MB vs {SIZE[REF]:.0f} MB; {PARAMS[DEPLOY]:.0f}M vs {PARAMS[REF]:.0f}M parameters).")
if pp["lo"] > -NONINF_DELTA:
    CLAIMS.append(f"Non-inferiority on PROMISE (single project-wise hold-out split, {len(SEEDS)} seed(s)): composite-score difference {pp['diff']:+.3f} [{pp['lo']:+.3f}, {pp['hi']:+.3f}], "
                  f"lower bound above the pre-specified margin -{NONINF_DELTA}.")
else:
    NOT_CLAIMS.append(f"Non-inferiority is NOT shown on test (difference {pp['diff']:+.3f} [{pp['lo']:+.3f}, {pp['hi']:+.3f}], margin -{NONINF_DELTA}): report it as a trade-off.")
if pp["lo"] > 0 and (np.isnan(p_h) or p_h < 0.05): CLAIMS.append(f"Superiority over the baseline is supported (Holm p={p_h:.3f}).")
else: NOT_CLAIMS.append("Do NOT claim the compressed model is more accurate than the baseline: the paired CI includes 0 (or Holm p >= 0.05).")
if len(SEEDS) < 3: NOT_CLAIMS.append(f"Only {len(SEEDS)} training seed(s): run-to-run variability is not established (use >= 3).")
if EXT is not None:
    CLAIMS.append(f"External dataset ({EXTERNAL_DATASET['name']}, n={EXT['n']}): difference {EXT['delta']:+.3f} [{EXT['lo']:+.3f}, {EXT['hi']:+.3f}]; "
                  + ("non-inferior within the same margin." if EXT["noninferior"] else "NOT shown non-inferior."))
else: NOT_CLAIMS.append("No second dataset: generalisation beyond PROMISE is untested; state this as a limitation.")
if HUMAN_RESULTS is None: NOT_CLAIMS.append("No human evaluation: make NO claim about explanation quality (only format/consistency numbers exist).")
else:
    kmin = min(v for v in HUMAN_RESULTS["kappa"].values() if not np.isnan(v))
    if kmin < 0.4: NOT_CLAIMS.append(f"Human ratings exist but agreement is too low (minimum kappa {kmin:.2f} < 0.40): make no explanation-quality claim until guidelines are refined and re-rated.")
    else: CLAIMS.append(f"Human-rated explanation quality (n={HUMAN_RESULTS['n_items']} items, min kappa {kmin:.2f}): deployment minus baseline "
                        f"{HUMAN_RESULTS['deploy_minus_baseline_expl']:+.2f} [{HUMAN_RESULTS['ci'][0]:+.2f}, {HUMAN_RESULTS['ci'][1]:+.2f}] on a 1-5 scale.")
NOT_CLAIMS.append("Do NOT compare numbers with the NICE paper as if they were the same experiment (different split, labels' version, LLM for targets); call this a re-implementation.")
NOT_CLAIMS.append("Do NOT describe the explanations as the model's reasoning: they imitate LLM rationales written from gold labels.")
if LEAK_DELTA is not None:
    if LEAK_DELTA["lo"] > 0: CLAIMS.append(f"Methodological finding: random splits inflate the (TF-IDF classifier) score by {LEAK_DELTA['diff']:+.3f} [{LEAK_DELTA['lo']:+.3f}, {LEAK_DELTA['hi']:+.3f}] versus project-wise splits.")
    else: NOT_CLAIMS.append(f"Leakage inflation is not statistically established (difference {LEAK_DELTA['diff']:+.3f} [{LEAK_DELTA['lo']:+.3f}, {LEAK_DELTA['hi']:+.3f}]): do not present it as a finding.")
print("SUPPORTED CLAIMS"); [print("  +", c) for c in CLAIMS]
print("\nNOT SUPPORTED / MUST BE QUALIFIED"); [print("  -", c) for c in NOT_CLAIMS]


SUPPORTED CLAIMS
  + Compression: baseline [int8-safe] is 1.9x smaller than the google/flan-t5-base baseline (509 MB vs 990 MB; 248M vs 248M parameters).
  + Non-inferiority on PROMISE (single project-wise hold-out split, 1 seed(s)): composite-score difference +0.001 [-0.010, +0.010], lower bound above the pre-specified margin -0.03.
  + Methodological finding: random splits inflate the (TF-IDF classifier) score by +0.167 [+0.132, +0.204] versus project-wise splits.

NOT SUPPORTED / MUST BE QUALIFIED
  - Do NOT claim the compressed model is more accurate than the baseline: the paired CI includes 0 (or Holm p >= 0.05).
  - Only 1 training seed(s): run-to-run variability is not established (use >= 3).
  - No second dataset: generalisation beyond PROMISE is untested; state this as a limitation.
  - No human evaluation: make NO claim about explanation quality (only format/consistency numbers exist).
  - Do NOT compare numbers with the NICE paper as if they were the same experiment (differe

[None, None, None, None, None, None]

In [29]:
from transformers import AutoConfig
EXPORT = os.path.join(WORK_DIR, "export"); shutil.rmtree(EXPORT, ignore_errors=True); os.makedirs(EXPORT)
def save(x, name):
    if x is not None and len(x): x.to_csv(os.path.join(EXPORT, name), index=False)
save(headline, "results_test_pooled.csv"); save(DEPLOY_TAB, "selection_noninferiority_val.csv"); save(abl_kd, "ablation_kd.csv"); save(abl_q, "ablation_quantization.csv")
save(abl_ext, "comparison_external.csv"); save(proj_tab, "per_project.csv"); save(err_df, "errors_deployment_model.csv"); save(expl_tab, "explanation_metrics.csv")
save(judge_tab, "llm_judge.csv"); save(lat_tab, "latency.csv")
save(abl_expl, "ablation_explanations.csv"); save(LR_SWEEP_TAB, "lr_search_validation.csv"); save(LR_CHOSEN_TAB, "lr_search_chosen.csv")
save(SEED_TAB.astype(str), "seed_variability.csv"); save(LEAK_TAB, "leakage_study.csv"); save(HUMAN_TAB, "human_eval_summary.csv")
if EXT is not None: save(EXT["table"], "external_dataset_results.csv")
if os.path.isdir(HUMAN_EVAL_DIR): shutil.copytree(HUMAN_EVAL_DIR, os.path.join(EXPORT, "human_eval"), dirs_exist_ok=True)
open(os.path.join(EXPORT, "claims.txt"), "w").write("SUPPORTED CLAIMS\n" + "\n".join(CLAIMS) + "\n\nNOT SUPPORTED / MUST BE QUALIFIED\n" + "\n".join(NOT_CLAIMS) + "\n")
for m, t in pc_tab.items(): t.to_csv(os.path.join(EXPORT, "per_class_" + re.sub(r"[^A-Za-z0-9]+", "_", m).strip("_") + ".csv"))
if EXPL_VALID and "HUMAN_SHEET" in globals(): save(HUMAN_SHEET, "human_eval_sheet_blinded.csv"); save(HUMAN_KEY, "human_eval_KEY_do_not_share.csv")
shutil.copy(os.path.join(WORK_DIR, "pareto.png"), EXPORT)
pd.DataFrame({"requirement": test_all.text, "project": test_all.project, "outer_fold": test_all.outer, "gold": test_all.tgt,
              "baseline_pred": ARR[REF]["test"][5], "deployment_pred": ARR[DEPLOY]["test"][5]}).to_csv(os.path.join(EXPORT, "predictions_test_oof.csv"), index=False)
os.makedirs(os.path.join(EXPORT, "splits"))
for k in FOLDS:
    for nm in ("train", "val", "test"): R[(k, S0)][nm + "_df"].drop(columns=["src", "tgt"], errors="ignore").to_csv(os.path.join(EXPORT, "splits", f"fold{k}_{nm}.csv"), index=False)
json.dump(dict(protocol="single project-wise train/validation/test split", folds={k: R[(k, S0)]["leak"] for k in FOLDS},
               model_selection="validation predictions only; non-inferiority margin pre-specified", delta=NONINF_DELTA, bootstrap=dict(unit=BOOT_UNIT, B=BOOT_N),
               explanation_note="explanations are generated from gold labels and used only as training targets / references, never as input",
               explanations_valid=EXPL_VALID, train_seeds=SEEDS, lr_grid=dict(baseline=list(BASE_LR_GRID), student_mults=list(STUDENT_LR_MULTS)), refit=REFIT_INFO, external_dataset=(EXTERNAL_DATASET.get("name") if EXT is not None else None), human_eval_done=HUMAN_RESULTS is not None, explanation_writer_model=WRITER_MODEL if EXPL_VALID else None, judge_model=(JUDGE_MODEL_USED if RUN_LLM_JUDGE else None), dataset_md5_ok=DATA_MD5_OK),
          open(os.path.join(EXPORT, "protocol_and_leakage_report.json"), "w"), indent=1, default=str)
json.dump(dict(PRESET=PRESET, SEED=SEED, FOLDS=FOLDS, SPLIT=SPLIT, LEAK_FOLDS=LEAK_FOLDS, CFG={k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG.items()},
               STUDENTS=STUDENTS, BASELINE=BASELINE_NAME, TEACHER=TEACHER_NAME, KD=dict(alpha=KD_ALPHA, T=KD_T), NEAR_DUP=NEAR_DUP_THRESH,
               OVERSAMPLE=[OVERSAMPLE_RARE, MIN_CLASS_COUNT], MAX_LEN=[MAX_IN_LEN, MAX_OUT_LEN], ENV=ENV, DEPLOY=DEPLOY, REF=REF),
          open(os.path.join(EXPORT, "config_and_environment.json"), "w"), indent=1, default=str)
for f in os.listdir(WORK_DIR):
    if f.startswith("explanations_") and f.endswith(".json"): shutil.copy(os.path.join(WORK_DIR, f), EXPORT)

# ---------- release the deployment model (trained on 4/5 of the projects; fold with the best validation score)
LOADER_PY = '''"""Load the compressed NICE model:  from load_model import load; model, tok = load()"""
import os, torch
from transformers import AutoConfig, AutoModelForSeq2SeqLM, AutoTokenizer
from torch.ao.quantization import quantize_dynamic, default_dynamic_qconfig, float_qparams_weight_only_qconfig
HERE = os.path.dirname(os.path.abspath(__file__)); KIND = "__KIND__"

def load():
    tok = AutoTokenizer.from_pretrained(HERE)
    if KIND in ("fp32", "nf4"):
        return AutoModelForSeq2SeqLM.from_pretrained(HERE, device_map="auto" if KIND == "nf4" else None).eval(), tok
    m = AutoModelForSeq2SeqLM.from_config(AutoConfig.from_pretrained(HERE)).eval()
    if KIND == "int8-safe":
        spec = {n: default_dynamic_qconfig for n, mod in m.named_modules()
                if isinstance(mod, torch.nn.Linear) and not (n.endswith("lm_head") or n.endswith(".wo"))}
    else:
        spec = {torch.nn.Linear: default_dynamic_qconfig}
        if KIND == "int8+emb": spec[torch.nn.Embedding] = float_qparams_weight_only_qconfig
    q = quantize_dynamic(m, spec, dtype=torch.qint8)
    if KIND == "int8+emb": q.encoder.embed_tokens = q.shared; q.decoder.embed_tokens = q.shared
    q.load_state_dict(torch.load(os.path.join(HERE, "model_int8.pt"), weights_only=False))
    return q, tok

if __name__ == "__main__":
    model, tok = load()
    text = "Identify whether the software requirement is functional and/or quality (non-functional), classify its NFR categories, and explain.\\nRequirement: The system shall respond within 2 seconds."
    enc = tok(text, return_tensors="pt").to(model.device)
    print(tok.decode(model.generate(**enc, max_new_tokens=160, num_beams=4)[0], skip_special_tokens=True))
'''
dc = R[(FOLDS[0], S0)]["cands"][DEPLOY]; kind = dc["kind"]
best_k = max(FOLDS, key=lambda k: score_raw(R[(k, S0)]["cands"][DEPLOY]["val_raw"], R[(k, S0)]["val_df"])["score"])
src_path = REFIT_PATH if REFIT_PATH else R[(best_k, S0)]["cands"][DEPLOY]["path"]; cm_dir = os.path.join(EXPORT, "compressed_model"); os.makedirs(cm_dir)
if src_path and os.path.isdir(src_path):
    if kind == "fp32": shutil.copytree(src_path, cm_dir, dirs_exist_ok=True)
    elif kind == "nf4": m_ = load_nf4(src_path); m_.save_pretrained(cm_dir); tok.save_pretrained(cm_dir); del m_
    else:
        m_ = load_int8(src_path, {"int8": "plain", "int8-safe": "safe", "int8+emb": "emb"}[kind]); torch.save(m_.state_dict(), os.path.join(cm_dir, "model_int8.pt"))
        AutoConfig.from_pretrained(src_path).save_pretrained(cm_dir); tok.save_pretrained(cm_dir); del m_
    open(os.path.join(cm_dir, "load_model.py"), "w").write(LOADER_PY.replace("__KIND__", kind))
    shutil.copytree(src_path, os.path.join(EXPORT, "source_fp32_model"), dirs_exist_ok=True)
else: print("Deployment weights not found on disk (KEEP_MODELS=False?) -> model not exported.")

pdp = paired(DEPLOY, REF)
open(os.path.join(EXPORT, "README.txt"), "w").write(
    f"NICE compressed model (v3)\n==========================\nDeployment choice: {DEPLOY}  ({'REFIT on all ' + str(REFIT_INFO['projects']) + ' projects' if REFIT_PATH else 'weights from outer fold ' + str(best_k) + '; trained on ~4/5 of the projects, NOT refit'})\n"
    f"Size: {SIZE[DEPLOY]:.1f} MB vs baseline {SIZE[REF]:.1f} MB ({SIZE[REF] / SIZE[DEPLOY]:.1f}x smaller)\n"
    f"Held-out project-wise test score: {fmt_ci(PT[DEPLOY]['score'], BT[DEPLOY]['score'])} (baseline {fmt_ci(PT[REF]['score'], BT[REF]['score'])})\n"
    f"Paired delta vs baseline: {pdp['diff']:+.3f} [{pdp['lo']:+.3f}, {pdp['hi']:+.3f}]; non-inferiority margin {NONINF_DELTA}\n"
    f"Explanations valid: {EXPL_VALID}\nSee protocol_and_leakage_report.json and config_and_environment.json for the full protocol.\n")
zip_path = shutil.make_archive(os.path.join(WORK_DIR, "NICE_v2_outputs"), "zip", EXPORT)
print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
try:
    from google.colab import files; files.download(zip_path)
except ImportError: print("Not in Colab - the zip is in the working directory (Kaggle: /kaggle/working; use the Output tab).")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Created /kaggle/working/nice_conference_efficient/NICE_v2_outputs.zip (1357 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Limitations to state in the paper (and what is still needed before submission)

**Now handled in code (v3):** a single training seed (42); learning-rate search per fold on validation projects; refit of the released model on all projects; explicit claim rules; leakage study; ablations (KD, quantizer, explanation-free); human-evaluation analysis (kappa, CIs, gold control); optional external dataset.

**Still limited — say so explicitly:**
1. **The second dataset is only as good as what you provide.** If `EXTERNAL_DATASET` is not configured, generalisation beyond PROMISE is untested. Label taxonomies rarely match exactly: only shared classes are scored, so state which. The PURE requirements corpus has no official NFR-category labels; use a labelled derivative or an industrial set whose labels you can map.
2. **Human evaluation needs people.** Until >= 2 annotators fill `human_eval/sheet_blank.csv` and the analysis cell is re-run, make **no claim about explanation quality**. Report kappa; if it is below 0.4, refine the guidelines.
3. **Explanations are post-hoc rationales.** Targets are written by an LLM from the gold labels; a model can imitate them without "reasoning". Report them as such, and rely on human ratings, not on ROUGE/BERTScore.
4. **The LLM judge is supporting evidence only.** Even with a different Groq model than the reference writer it is still an LLM judge served by the same provider; report judge-human correlation.
5. **Hyper-parameter search is limited:** learning rate only (grid of 3), teacher learning rate and KD (alpha, temperature) are fixed, and the search runs on the first seed. Report the sensitivity table.
6. **Refit cannot be evaluated on held-out data** (everything was used for training). Its expected quality is the CV estimate of the same recipe; the only independent check is the external dataset.
7. **Near-duplicate removal (0.80 TF-IDF cosine) is a heuristic;** cross-project paraphrases can remain. Report the number removed per fold (leakage report).
8. **Leakage study uses a cheap classifier;** state this. Repeat with the neural baseline if you want to claim the same inflation for it.
9. **Latency** was measured on one machine, 30 requests, batch size 1; report hardware and threads (config export).
10. **The non-inferiority margin** (`NONINF_DELTA = 0.03`) is a design choice: justify it (e.g. relative to seed sd, shown in the seed table) and **never change it after seeing test results**.
11. **Novelty is a framing question, not a code question.** This remains a re-implementation with an extended, rigorous compression study. Position it that way (evaluation rigour, leakage quantification, compression trade-offs) and compare to the NICE paper only qualitatively.

**Suggested reporting.** Main table = `results_test_pooled.csv`; figure = `pareto.png`; ablations = `ablation_kd.csv`, `ablation_quantization.csv`, `ablation_explanations.csv`; hyper-parameter and seed tables; leakage table; external and human tables when available; `claims.txt`; protocol and leakage report as an appendix. The `quick` preset must never be reported.
